# QDT_SPDC — Multimode SPDC Digital Twin

This notebook provides a documented, self-contained implementation of the calculations supporting **“Analytical Benchmarks for a Multimode SPDC Digital Twin”**.

Two benchmark branches are implemented:

- **State level:** thermal loss acts directly on the SPDC modes; global purity and squared target-state fidelity are evaluated.
- **Measurement level:** the SPDC state is transformed by a lossless NPBS, post-NPBS thermal-loss channels act on the idler/signal arms, and polarization-resolved singles, coincidences, visibility, and conditional CHSH quantities are evaluated.

The polarization–spectral mode ordering is

$$
(H,f_0),(V,f_0),
\left\{(H,f_k^+),(V,f_k^-),(H,f_k^-),(V,f_k^+)\right\}_{k=1}^{N_k},
$$

with

$$
N_{\rm mode}=2+4N_k,
\qquad
M=1+2N_k.
$$

Here $N_k$ is the **integer sideband-pair count** in the discrete digital twin. It is distinct from the spectral Schmidt number.

## 1. Environment

Install the dependencies once in a clean Python environment using the repository `requirements.txt`:

```bash
pip install -r requirements.txt
```

The notebook uses QuTiP for finite-Fock quantum states and operators, `qutip-qip` for subsystem operator expansion, and `mpltern` for ternary phase maps.

In [ ]:
# ============================================================
# Standard library
# ============================================================
import csv
import itertools
import math
import multiprocessing as mp
import os
import re
import time
from collections import defaultdict
from datetime import datetime
from functools import lru_cache
from multiprocessing.dummy import Pool as ThreadPool
from typing import List, Tuple

# ============================================================
# Numerical / scientific computing
# ============================================================
import numpy as np
import pandas as pd
import sympy as sp
import scipy.linalg
from numpy.linalg import det, inv
from scipy.linalg import eigvals, expm, sqrtm, svd
from scipy.optimize import curve_fit, lsq_linear, minimize, minimize_scalar

# ============================================================
# Plotting
# ============================================================
import matplotlib as mpl
import matplotlib.cm as cm
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib import colors, colormaps
from matplotlib.colors import Normalize
from matplotlib.image import AxesImage
from matplotlib.ticker import (
    AutoMinorLocator,
    FuncFormatter,
    LogLocator,
    NullFormatter,
)
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d import Axes3D

# ============================================================
# Parallel processing
# ============================================================
from joblib import Parallel, delayed
from pathos.multiprocessing import ProcessingPool as Pool

# ============================================================
# Jupyter / display tools
# ============================================================
import ipywidgets as widgets
from IPython.display import Math, display
from rich import print

# ============================================================
# Ternary plotting
# ============================================================
import mpltern

# ============================================================
# QuTiP
# ============================================================
import qutip as qt
from qutip import Qobj, fidelity
from qutip_qip.operations import expand_operator

print(f"QuTiP version: {qt.__version__}")

## 2. Plotting and output configuration

Publication-style Matplotlib settings are defined here. Generated plots are written to `images/`, while numerical fields and contour paths used for figure reproduction are written to `csv_data/`.

In [ ]:
# Figure saving
savefigures = True

# General figure settings
plt.rcParams["text.usetex"] = False
plt.rcParams["figure.dpi"] = 100
plt.rcParams["pdf.fonttype"] = 42

# Fonts
mpl.rcParams["font.family"] = "serif"
cmfont = font_manager.FontProperties(fname=mpl.get_data_path() + "/fonts/ttf/cmr10.ttf")
mpl.rcParams["font.serif"] = cmfont.get_name()
mpl.rcParams["font.size"] = 14

# Math fonts
mpl.rcParams["mathtext.fontset"] = "cm"
mpl.rcParams["axes.formatter.use_mathtext"] = True
mpl.rcParams["axes.unicode_minus"] = False

# Axis label and tick sizes
mpl.rcParams["axes.labelsize"] = 14
mpl.rcParams["xtick.labelsize"] = 14
mpl.rcParams["ytick.labelsize"] = 14

# Legend font sizes
mpl.rcParams["legend.fontsize"] = 14
mpl.rcParams["legend.title_fontsize"] = 14

# Legend layout
mpl.rcParams["legend.loc"] = "best"
mpl.rcParams["legend.numpoints"] = 1
mpl.rcParams["legend.scatterpoints"] = 1
mpl.rcParams["legend.handlelength"] = 1
mpl.rcParams["legend.handletextpad"] = 0.4
mpl.rcParams["legend.borderpad"] = 0.4
mpl.rcParams["legend.columnspacing"] = 1
mpl.rcParams["legend.labelspacing"] = 0.4
mpl.rcParams["legend.borderaxespad"] = 0.5
mpl.rcParams["legend.markerscale"] = 1

# Legend frame
mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.facecolor"] = "white"
mpl.rcParams["legend.edgecolor"] = "black"
mpl.rcParams["legend.fancybox"] = False
mpl.rcParams["legend.framealpha"] = 1
mpl.rcParams["legend.shadow"] = False




os.makedirs('images', exist_ok=True)
os.makedirs('csv_data', exist_ok=True)

## 3. Symbolic and density-matrix visualization helpers

These utilities convert numerical states to rounded SymPy matrices, generate compact Fock-basis expressions, and visualize sparse density matrices.

For a multimode density operator,

$$
\rho=\sum_{ij}\rho_{ij}|i\rangle\langle j|,
$$

the 3D visualization encodes $|\rho_{ij}|$: diagonal terms are populations and off-diagonal terms are coherence magnitudes. The phase of a coherence is not represented by the bar height.

In [ ]:
import re
import itertools
from collections import defaultdict

import numpy as np
import sympy as sp

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os

'''--------State and density matrixes representations -----------------------------'''

def to_sympy_rounded(matrix, precision):
    return sp.Matrix(np.round(matrix, precision))

def generate_symbolic_state_or_rho(state, Fk_dim, precision, N_k, n_sp, mode_type):
    def extract_ket_bra(label):
        ket_match = re.search(r'\\left\|(.*?)\\right', label)
        bra_match = re.search(r'\\left\\langle(.*?)\\right', label)
        if ket_match and bra_match:
            return ket_match.group(1), bra_match.group(1)
        else:
            return None, None
    coeff_map = defaultdict(list)
    n_modes = 2 + int(N_k) * 4 + int(n_sp)
    mode_labels = []
    if mode_type == "polarized+spectural":
        mode_labels = ["H,f0", "V,f0"]
        for k in range(1, N_k + 1):
            mode_labels += [f"H,f{k}+", f"V,f{k}-", f"H,f{k}-", f"V,f{k}+"]
    elif mode_type == "polarized+spectural+spatial":
        mode_labels = ["H_i,f0", "V_i,f0", "H_s,f0", "V_s,f0"]
        for k in range(1, N_k + 1):
            mode_labels += [f"H_i,f{k}+", f"V_i,f{k}-", f"H_i,f{k}-", f"V_i,f{k}+",
                            f"H_s,f{k}+", f"V_s,f{k}-", f"H_s,f{k}-", f"V_s,f{k}+"]
    elif mode_type == "polarized+spectural+idler":
        mode_labels = ["H_i,f0", "V_i,f0"]
        for k in range(1, N_k + 1):
            mode_labels += [f"H_i,f{k}+", f"V_i,f{k}-", f"H_i,f{k}-", f"V_i,f{k}+"]
    elif mode_type == "polarized+spectural+signal":
        mode_labels = ["H_s,f0", "V_s,f0"]
        for k in range(1, N_k + 1):
            mode_labels += [f"H_s,f{k}+", f"V_s,f{k}-", f"H_s,f{k}-", f"V_s,f{k}+"]

    is_density_matrix = not (state.isket or state.isbra)
    for i, coeff in np.ndenumerate(state.full()):
        if abs(coeff) >= 10 ** (-2*precision):
            r = round(coeff.real, precision)
            im = round(coeff.imag, precision)
            coeff_key = (r, im)
            if is_density_matrix:
                row_basis = [(i[0] // Fk_dim ** k) % Fk_dim for k in range(n_modes)]
                col_basis = [(i[1] // Fk_dim ** k) % Fk_dim for k in range(n_modes)]
                row_label = ",".join([f"{row_basis[j]}_{{{mode_labels[j]}}}" for j in range(n_modes)])
                col_label = ",".join([f"{col_basis[j]}_{{{mode_labels[j]}}}" for j in range(n_modes)])
                label_str = f"\\left|{row_label}\\right\\rangle\\left\\langle{col_label}\\right|"
            else:
                basis_state = [(i[0] // Fk_dim ** k) % Fk_dim for k in range(n_modes)]
                label_str = ",".join([f"{basis_state[j]}_{{{mode_labels[j]}}}" for j in range(n_modes)])
                label_str = f"\\left|{label_str}\\right\\rangle"
            coeff_map[coeff_key].append(label_str)

    symbolic_terms = []
    used = set()
    coeff_items = list(coeff_map.items())

    for (r, im), basis_list in coeff_items:
        key = (r, im)
        if key in used:
            continue

        # Check for conjugate terms only if it's complex (not real or purely imaginary)
        is_conjugate_pair = (r, im) != (-r, -im) and (-r, -im) in coeff_map
        if is_conjugate_pair:
            basis1 = coeff_map[key]
            basis2 = coeff_map[(-r, -im)]
            paired = []
            used_pairs = set()

            for b1 in basis1:
                for b2 in basis2:
                    if (b1, b2) in used_pairs or (b2, b1) in used_pairs:
                        continue
                    b1_ket, b1_bra = extract_ket_bra(b1)
                    b2_ket, b2_bra = extract_ket_bra(b2)
                    if b1_ket is None or b2_ket is None:
                        continue
                    if b1_bra == b2_ket and b1_ket == b2_bra:
                        ordered = sorted([(b1_ket, b1_bra), (b2_ket, b2_bra)], reverse=True)
                        term1 = f"\\left|{ordered[0][0]}\\right\\rangle\\left\\langle{ordered[0][1]}\\right|"
                        term2 = f"\\left|{ordered[1][0]}\\right\\rangle\\left\\langle{ordered[1][1]}\\right|"
                        sign = "-" if im != 0 else "+"
                        combined = f"{term1} {sign} {term2}"
                        paired.append(combined)
                        used_pairs.add((b1, b2))

            if paired:
                coeff_value = f"{abs(im):.{precision}f}i" if im != 0 else f"{abs(r):.{precision}f}"
                term_body = " + \\\\\n".join(paired)
                term_str = f" + {coeff_value}\\left(\\begin{{aligned}}{term_body}\\end{{aligned}}\\right)"
                symbolic_terms.append(term_str)
                used.add(key)
                used.add((-r, -im))
            continue

        # Otherwise, this is a lone term (either real or non-paired complex)
        coeff_value = ""
        if im == 0:
            coeff_value = f"{r:+.{precision}f}"
        elif r == 0:
            coeff_value = f"{im:+.{precision}f}i"
        else:
            coeff_value = f"({r:+.{precision}f}{im:+.{precision}f}i)"

        prefix = "" if len(symbolic_terms) == 0 else ("  " if not coeff_value.startswith("-") else " ")
        if len(basis_list) == 1:
            term_str = f"{prefix}{coeff_value}{basis_list[0]}"
        else:
            grouped = " + \\\\\n".join(basis_list)
            term_str = f"{prefix}{coeff_value}\\left(\\begin{{aligned}}{grouped}\\end{{aligned}}\\right)"
        symbolic_terms.append(term_str)
        used.add(key)
    return "".join(symbolic_terms)



# Density matrix represenation
def plot_sparse_density_matrix(rho, total_modes, Fk_dim, N_k=0, threshold=1e-26,cmap_name = 'jet',save_path=None):
    dim = rho.shape[0]
    if N_k == 0:
        fig = plt.figure(figsize=(7, 9))
        fontsize_xticks, fontsize_yticks, fontsize_tick_params = 9, 9, 12
        pad_x, pad_y, pad_z = 6, 0, 3
        fig_title_size, colorbar_pad, colorbar_size = 9, 0.12, 11
        shrink_size, aspect_size, colorbar_labelsize = 0.25, 15, 12
    else:
        fig = plt.figure(figsize=(20, 24))
        fontsize_xticks, fontsize_yticks, fontsize_tick_params = 15, 15, 15
        pad_x, pad_y, pad_z = 27, 4, 5
        fig_title_size, colorbar_pad, colorbar_size = 15, 0.15, 20
        shrink_size, aspect_size, colorbar_labelsize = 0.25, 23, 15

    ax = fig.add_subplot(111, projection='3d')
    xpos, ypos, zpos, dx, dy, dz = [], [], [], [], [], []
    for i in range(dim):
        for j in range(dim):
            xpos.append(j)
            ypos.append(i)
            zpos.append(0)
            dx.append(0)
            dy.append(0)
            dz.append(np.abs(rho[i, j]))
    if not dz:
        dz = [0]

    norm = plt.Normalize(vmin=0.0, vmax=1.0)
    colors = plt.colormaps[cmap_name](norm(dz))
    dx = dy = 0.6
    xpos = [x + (1 - dx) / 2 for x in xpos]
    ypos = [y + (1 - dy) / 2 for y in ypos]

    ax.bar3d(xpos, ypos, zpos, dx, dy, dz, color=colors, alpha=1)
    mappable = cm.ScalarMappable(norm=norm, cmap=cmap_name)
    mappable.set_array([])
    cbar = fig.colorbar(mappable, shrink=shrink_size, aspect=aspect_size, pad=1.0*colorbar_pad, ax=ax)
    cbar.set_label(r"$|\rho_{ij}|$", fontsize=colorbar_size)
    cbar.ax.tick_params(labelsize=colorbar_labelsize)

    basis_states = list(itertools.product(range(Fk_dim), repeat=total_modes))
    full_labels = [rf"$\left|{','.join(map(str, b))}\right\rangle$" for b in basis_states]

    if N_k == 0:
        xtick_labels = ytick_labels = full_labels
        xtick_positions = [x + 0.5 for x in range(len(full_labels))]
        ytick_positions = [y + 0.5 for y in range(len(full_labels))]
    else:
        xtick_labels, ytick_labels, xtick_positions, ytick_positions = [], [], [], []
        for i in range(dim):
            if np.any(np.abs(rho[i, :]) > threshold) or np.any(np.abs(rho[:, i]) > threshold):
                xtick_positions.append(i + 0.5)
                ytick_positions.append(i + 0.5)
                xtick_labels.append(full_labels[i])
                ytick_labels.append(full_labels[i])

    ax.set_xticks(xtick_positions)
    ax.set_yticks(ytick_positions)
    ax.set_xticklabels(xtick_labels, rotation=65, fontsize=fontsize_xticks, ha='center', va='center')
    ax.tick_params(axis='x', labelsize=fontsize_tick_params, pad=pad_x)
    ax.set_xlim(0, dim)
    ax.set_yticklabels(ytick_labels, fontsize=fontsize_yticks, ha='left', va='center')
    ax.tick_params(axis='y', labelsize=fontsize_tick_params, pad=pad_y)
    ax.set_ylim(0, dim)
    ax.set_zlim(0, 1)
    ax.set_zticks([0, 0.25, 0.5, 0.75, 1])
    ax.set_zticklabels(['0', '0.25', '0.5', '0.75', '1'])
    ax.tick_params(axis='z', labelsize=fontsize_tick_params, pad=pad_z)
    #ax.set_zlabel("Probability", fontsize=fontsize_tick_params, labelpad=1 * pad_z)
    ax.set_title("Mixed-state Density Matrix", fontsize=2 * fig_title_size)
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 4. Core SPDC, NPBS, and thermal-loss primitives

The finite-Fock digital twin is assembled from four basic operations.

### SPDC Hamiltonian

The source contains $M=1+2N_k$ mutually disjoint conjugate mode pairs. In the infinite-dimensional description,

$$
\hat H_{\rm SPDC}
=
i\hbar\kappa\sum_{m=1}^{M}
\left(
e^{i\phi_m}\hat a_m^\dagger\hat b_m^\dagger
-
e^{-i\phi_m}\hat a_m\hat b_m
\right).
$$

The numerical code uses an equivalent Hermitian pair-creation convention.

### Thermal-loss channel

Each system mode is coupled to an independent thermal ancilla through a virtual beam splitter,

$$
\hat a_{\rm out}
=
\sqrt{T}\,\hat a_{\rm in}
+
\sqrt{1-T}\,\hat e,
\qquad
\langle\hat e^\dagger\hat e\rangle=\bar n_{\rm th},
$$

then the environmental output is traced out.

The helper functions below also provide photon-pair statistics, auto/cross $g^{(2)}$, fidelity, and Wigner-function visualization.

In [ ]:
import numpy as np
from scipy.linalg import eigvals
import matplotlib.pyplot as plt
import qutip as qt
from qutip_qip.operations import expand_operator
import os


def compute_precision(Fk_dim, g, N_k):
    D = Fk_dim ** (2 + 4 * N_k)
    max_n = np.sqrt(D) - 1
    p_min = np.tanh(g) ** (2 * max_n)
    epsilon = 1e-100
    precision = min(12, max(4, int(-np.log10(p_min + epsilon) + 2)))
    return precision


#Hamiltonian SPDC: all pair correlations between modes with orthogonal polarization only
def hamiltonian_spdc(N_k, Fk_dim, g_spdc):
    '''Mode structure:
    - Index 0: H, f0
    - Index 1: V, f0
    - For k in 0 to N_k-1:
        Index 2 + 4k:     H, f_k^+
        Index 2 + 4k + 1: V, f_k^-
        Index 2 + 4k + 2: H, f_k^-
        Index 2 + 4k + 3: V, f_k^+ '''
    mode_labels = [("H", "f0"), ("V", "f0")]
    for k in range(N_k):
        mode_labels += [("H", f"f{k+1}+"), ("V", f"f{k+1}-"), ("H", f"f{k+1}-"), ("V", f"f{k+1}+")]
    total_modes = len(mode_labels)
    # Annihilation operators:
    def a(m):
        ops = [qt.qeye(Fk_dim) for _ in range(total_modes)]
        ops[m] = qt.destroy(Fk_dim)
        return qt.tensor(*ops)
    a_ops = [a(i) for i in range(total_modes)]
    # Energy conservation (e.g., fk+ + fk− =const)
    # Momentum conservation (phase-matching with conjugate frequencies/ signal and idler are created in opposite k modes, or at least k-conjugate pairs)
    H_spdc = 0
    def match(f1, f2):
        return (f1 == f2 == "f0") or (f1[:-1] == f2[:-1] and {f1[-1], f2[-1]} == {"+", "-"})
    # Add SPDC terms only for orthogonal polarization pairs
    for i in range(total_modes):
        for j in range(i + 1, total_modes):  # i < j to ensure Mode symmetry (e.g., indistinguishability, bosonic symmetrization)
            pol_i, freq_i = mode_labels[i]
            pol_j, freq_j = mode_labels[j]
            if pol_i != pol_j and match(freq_i, freq_j):  # Polarization orthogonality (H-V pairs) and momentum-Energy conservation
                H_pair = g_spdc * (a_ops[i].dag() @ a_ops[j].dag())
                H_spdc += H_pair + H_pair.dag()

    return H_spdc


# Hamiltonian non polarizing beam splitter (NPBS)
def hamiltonian_np_beamsplitter(Fk_dim,T_bs,phi_bs):
    theta = np.arccos(np.sqrt(T_bs))
    a = qt.destroy(Fk_dim) & qt.qeye(Fk_dim)
    b = qt.qeye(Fk_dim) & qt.destroy(Fk_dim)
    H_bs = -1j * np.exp(1j * phi_bs) * a.dag() @ b
    H_bs += H_bs.dag()
    U_bs = (-1j * theta * H_bs).expm()
    return U_bs

# Non Polarizing Beam Spliter (NPBS)
def apply_NPBS(rho_in, Fk_dim, T_bs, nth_bs, phi_bs, N_k):
    total_modes = 2 + 4 * int(N_k)
    total_system_modes = 2 * total_modes
    rho_th_bs = qt.tensor(*[qt.thermal_dm(Fk_dim, nth_bs) for _ in range(total_modes)])
    rho_before_bs = rho_in & rho_th_bs
    U_full = qt.tensor(*[qt.qeye(Fk_dim) for _ in range(total_system_modes)])
    dims = [Fk_dim] * total_system_modes
    for i in range(total_modes):
        U_bs = hamiltonian_np_beamsplitter(Fk_dim, T_bs, phi_bs)
        U_i = expand_operator(U_bs, dims=dims, targets=[i, i + total_modes])
        U_full = U_i @ U_full
    rho_out = U_full @ rho_before_bs @ U_full.dag()
    return U_full,rho_out


# Hamiltonian Digital Twin: apply loss and thermal noise using virtual beamsplitters to a system of multiple modes
def apply_digital_twin_model_single_mode(rho_in, mode_idx, Fk_dim, T, nth, phi):
    U_dt = U_bs = hamiltonian_np_beamsplitter(Fk_dim,T,phi)
    total_modes = len(rho_in.dims[0])
    rho_th = qt.thermal_dm(Fk_dim, nth)
    # Step 1: Add thermal mode
    rho_ext = rho_th & rho_in  # thermal comes first temporarily
    # Step 2: Permute to bring thermal to correct place
    # Thermal at position 0, signal at position mode_idx+1
    perm = [0, mode_idx + 1] + [j for j in range(1, total_modes + 1) if j - 1 != mode_idx]
    rho_perm = rho_ext.permute(perm)
    # Step 3: Build full operator
    ops = [qt.qeye(Fk_dim)] * (total_modes)
    ops[0] = U_dt
    U_full = qt.tensor(*ops)
    # Step 4: Apply unitary and trace out thermal (index 0)
    rho_after_dt = U_full @ rho_perm @ U_full.dag()
    rho_out = rho_after_dt.ptrace([j for j in range(total_modes + 1) if j != 0])
    return rho_out



def Fidelity(rho1, rho2):
    M = rho1.full() @ rho2.full()
    lambdas = eigvals(M)
    s = np.sqrt(lambdas)
    return np.real(np.sum(s) ** 2)



# Phototn Statisctics
def count_photon_pairs(rho, basis_dims, pair_indices):
    prob_pairs = {}
    total_dim = np.prod(basis_dims)
    rho_diag = np.real(np.diag(rho.full()))
    for flat_idx in range(total_dim):
        prob = rho_diag[flat_idx]
        if prob == 0:
            continue
        idx_tuple = np.unravel_index(flat_idx, basis_dims)
        pair_count = sum(min(int(idx_tuple[i]), int(idx_tuple[j])) for i, j in pair_indices)
        prob_pairs[pair_count] = prob_pairs.get(pair_count, 0) + prob
    return prob_pairs



def compute_mode_statistics(rho, Fk_dim, total_modes):
    i = total_modes - 1
    identity_ops = [qt.qeye(Fk_dim) for _ in range(total_modes)]
    number_ops = [qt.tensor(*(identity_ops[:i] + [qt.num(Fk_dim)] + identity_ops[i+1:])) for i in range(total_modes)]
    n_avg = qt.expect(number_ops[i], rho)
    n_sq_avg = qt.expect(number_ops[i] ** 2, rho)
    std = np.sqrt(n_sq_avg - n_avg ** 2)
    distribution = rho.ptrace(i).diag().real
    return n_avg, std, distribution


# single-mode second-order autocorrelation at zero delay
def compute_g2_auto_mode(rho, Fk_dim, modes):
    rho_mode = rho.ptrace(modes)
    a = qt.destroy(Fk_dim)
    n = np.real(qt.expect(a.dag() @ a, rho_mode))
    numerator = np.real(qt.expect(a.dag() @ a.dag() @ a @ a, rho_mode))
    return np.nan if n < 1e-12 else float(numerator / n**2)
    
# correlation between the two generated photons rather than thermal statistics within one mode
def compute_g2_cross(rho, Fk_dim, mode_s=0, mode_i=1):
    rho_si = rho.ptrace([mode_s, mode_i])
    a_s = qt.tensor(qt.destroy(Fk_dim), qt.qeye(Fk_dim))
    a_i = qt.tensor(qt.qeye(Fk_dim), qt.destroy(Fk_dim))
    n_s = np.real(qt.expect(a_s.dag() @ a_s, rho_si))
    n_i = np.real(qt.expect(a_i.dag() @ a_i, rho_si))
    num = np.real(qt.expect(a_s.dag() @ a_i.dag() @ a_i @ a_s, rho_si))
    return np.nan if n_s*n_i < 1e-12 else float(num/(n_s*n_i))
    

def plot_wigner(rho, Fk_dim, save_path=None):
    xmax = max(5, np.sqrt(Fk_dim) * 1.5)
    xvec = np.linspace(-xmax, xmax, 200)
    W = qt.wigner(rho, xvec, xvec)
    fig, ax = plt.subplots(figsize=(3,3))
    im = ax.contourf(xvec, xvec, W, levels=80, cmap="jet")
    ax.set_xlabel(r"$x_H$")
    ax.set_ylabel(r"$x_V$")
    plt.tight_layout()
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 5. Composite finite-Fock state evolution

`state_evolution_spdc` combines the primitives into the configurable digital-twin workflow:

1. construct the multimode vacuum;
2. generate the SPDC state;
3. apply mode-by-mode thermal loss before the NPBS when requested;
4. evaluate purity, fidelity, photon statistics, $g^{(2)}$, Cauchy–Schwarz ratio, and von Neumann entropy;
5. apply the NPBS;
6. optionally apply post-NPBS thermal loss to the full output register or to a selected arm;
7. prepare symbolic and matrix representations for inspection.

The exact analytical benchmarks used later do **not** depend on these auxiliary symbolic diagnostics; they are retained to make the notebook a complete representation of the supplied finite-Fock workflow.

In [ ]:
def state_evolution_spdc(args):
    results = {}
    Fk_dim, N_k, g, precision,\
    T_dt_for_spdc,nth_dt_for_spdc,phi_dt_for_spdc,\
    T_bs, phi_bs, nth_bs,\
    T_dt_for_bs_full_mode,nth_dt_for_bs_full_mode,phi_dt_for_bs_full_mode,\
    T_dt_for_bs,nth_dt_for_bs,phi_dt_for_bs = args

    total_modes = 2 + 4 * N_k

    '''--------------------------- SPDC ------------------------------------ '''
    #SPDC pure entanglement states generation:
    psi0 = qt.tensor(*[qt.basis(Fk_dim, 0) for _ in range(total_modes)])
    rho0 = psi0 @ psi0.dag()
    H_spdc = qt.Qobj(hamiltonian_spdc(N_k, Fk_dim, g))
    U_spdc = (-1j * H_spdc).expm()
    psi_out_spdc = U_spdc @ psi0
    rho_out_spdc = U_spdc @ rho0 @ U_spdc.dag()

    #Apply digital twin (dt) model after SPDC proccess:
    rho_spdc_after_dt = []
    rho_spdc_in_dt = rho_out_spdc
    for i in range(total_modes):
        rho_spdc_in_dt = apply_digital_twin_model_single_mode(rho_spdc_in_dt, i, Fk_dim, T_dt_for_spdc, nth_dt_for_spdc, phi_dt_for_spdc)   # Keep P and F constant for similar mode index when it changes from n_k = 0 to 1
        rho_spdc_after_dt.append(rho_spdc_in_dt)

    rho_spdc_after_dt_final_state = rho_spdc_after_dt[-1]

    # Purity and Fidelity
    purities_spdc_after_dt = [(rho @ rho).tr() for rho in rho_spdc_after_dt]
    fidelities_spdc_after_dt = [Fidelity(rho,rho_out_spdc) for rho in rho_spdc_after_dt]   #qt.metrics.fidelity / qt.fidelity () / qutip.fidelity


    # Photon statistics:
    pair_mode_indices: List[Tuple[int, int]] = [(0, 1)]  # H_f0 and V_f0 pair
    for k in range(N_k):
        base = 2 + 4 * k
        pair_mode_indices.append((base, base + 1))      # H_f_k^+ and V_f_k^- pair
        pair_mode_indices.append((base + 2, base + 3))  # H_f_k^- and V_f_k^+ pair

    P_n = count_photon_pairs(rho_spdc_after_dt_final_state, rho_spdc_after_dt_final_state.dims[0], pair_mode_indices)
    mean, std, distribution = compute_mode_statistics(rho_spdc_after_dt_final_state, Fk_dim,total_modes)
    g2_ss = compute_g2_auto_mode(rho_spdc_after_dt_final_state, Fk_dim, 0)
    g2_ii = compute_g2_auto_mode(rho_spdc_after_dt_final_state, Fk_dim, 1)
    g2_si = compute_g2_cross(rho_spdc_after_dt_final_state, Fk_dim, 0, 1)

    # Cauchy–Schwarz violation (R > 1: non clasical correclation)
    R_cs = np.nan
    if np.isfinite(g2_ss) and np.isfinite(g2_ii) and np.isfinite(g2_si):
        if g2_ss*g2_ii > 1e-12:
            R_cs = g2_si**2/(g2_ss*g2_ii)

    # Entanglement von Neumann entropy (or mutual information)
    # The entropy is zero for pure states and reaches a maximum for maximally mixed states
    entropy_vn_spdc = float(qt.entropy_vn(rho_spdc_after_dt_final_state))

    '''------------------------------ NPBS --------------------------------- '''
    # Apply Non-Polarizing Beam Spliter (NPBS) on the mixed spdc state:
    U_bs, rho_spdc_after_bs = apply_NPBS(rho_spdc_after_dt_final_state, Fk_dim, T_bs, nth_bs, phi_bs, N_k)
    purity_spdc_after_bs = (rho_spdc_after_bs @ rho_spdc_after_bs).tr()
    fidelity_spdc_after_bs = Fidelity(rho_spdc_after_bs, rho_out_spdc & rho_out_spdc)
    #Apply digital twin (dt) model on the full-mode mixed state after BS:
    rho_spdc_after_bs_after_dt_full_mode = []
    rho_spdc_after_bs_in_dt_full_mode = rho_spdc_after_bs
    for i in range(2*total_modes):
        rho_spdc_after_bs_in_dt_full_mode = apply_digital_twin_model_single_mode(rho_spdc_after_bs_in_dt_full_mode, i, Fk_dim, T_dt_for_bs_full_mode, nth_dt_for_bs_full_mode, phi_dt_for_bs_full_mode)
        rho_spdc_after_bs_after_dt_full_mode.append(rho_spdc_after_bs_in_dt_full_mode)

    rho_spdc_after_bs_after_dt_full_mode_final_state = rho_spdc_after_bs_after_dt_full_mode[-1]
    purities_spdc_after_bs_after_dt_full_mode = [(rho @ rho).tr() for rho in rho_spdc_after_bs_after_dt_full_mode]
    fidelities_spdc_after_bs_after_dt_full_mode = [Fidelity(rho_spdc_after_bs,rho) for rho in rho_spdc_after_bs_after_dt_full_mode]

    '''------------------- idler and signal modes -------------------------- '''
    #Apply digital twin (dt) model on the mixed state after BS on idler/signal mode:
    rho_spdc_after_bs_idler = rho_spdc_after_bs.ptrace(list(range(total_modes)))
    rho_spdc_after_bs_signal = rho_spdc_after_bs.ptrace(list(range(total_modes, 2 * total_modes)))
    purity_spdc_after_bs_idler = (rho_spdc_after_bs_idler @ rho_spdc_after_bs_idler).tr()
    fidelity_spdc_after_bs_idler = Fidelity(rho_spdc_after_bs_idler, rho_out_spdc)
    purity_spdc_after_bs_signal = (rho_spdc_after_bs_signal @ rho_spdc_after_bs_signal).tr()
    fidelity_spdc_after_bs_signal = Fidelity(rho_spdc_after_bs_signal, rho_out_spdc)

    rho_spdc_after_bs_after_dt_idler = []
    rho_spdc_after_bs_in_dt = rho_spdc_after_bs_idler
    for i in range(total_modes):
        rho_spdc_after_bs_in_dt = apply_digital_twin_model_single_mode(rho_spdc_after_bs_in_dt, i, Fk_dim, T_dt_for_bs, nth_dt_for_bs, phi_dt_for_bs)
        rho_spdc_after_bs_after_dt_idler.append(rho_spdc_after_bs_in_dt)

    rho_spdc_after_bs_after_dt_idler_final_state = rho_spdc_after_bs_after_dt_idler[-1]

    purities_spdc_after_bs_after_dt_idler = [(rho @ rho).tr() for rho in rho_spdc_after_bs_after_dt_idler]
    fidelities_spdc_after_bs_after_dt_idler = [Fidelity(rho_out_spdc,rho) for rho in rho_spdc_after_bs_after_dt_idler]


    ''' ------------- Density matrixes representations ----------------------'''
    U_spdc_matrix = to_sympy_rounded(U_spdc.full(),precision)
    psi_out_spdc_matrix = to_sympy_rounded(psi_out_spdc.full(), precision)
    rho_out_spdc_matrix = to_sympy_rounded(rho_out_spdc.full(), precision)
    rho_spdc_after_dt_final_state_matrix = to_sympy_rounded(rho_spdc_after_dt_final_state.full(), precision)
    U_bs_matrix = to_sympy_rounded(U_bs.full(), precision)
    rho_spdc_after_bs_matrix = to_sympy_rounded(rho_spdc_after_bs.full(), precision)
    rho_spdc_after_bs_after_dt_full_mode_final_state_matrix = to_sympy_rounded(rho_spdc_after_bs_after_dt_full_mode_final_state.full(), precision)
    rho_spdc_after_bs_idler_matrix = to_sympy_rounded(rho_spdc_after_bs_idler.full(), precision)
    rho_spdc_after_bs_signal_matrix = to_sympy_rounded(rho_spdc_after_bs_signal.full(), precision)
    rho_spdc_after_bs_after_dt_idler_final_state_matrix = to_sympy_rounded(rho_spdc_after_bs_after_dt_idler_final_state.full(), precision)



    symbolic_U_spdc = generate_symbolic_state_or_rho(U_spdc, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural")
    symbolic_psi_out_spdc = generate_symbolic_state_or_rho(psi_out_spdc, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural")
    symbolic_rho_out_spdc = generate_symbolic_state_or_rho(rho_out_spdc, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural")
    symbolic_rho_spdc_after_dt_final_state = generate_symbolic_state_or_rho(rho_spdc_after_dt_final_state, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural")
    symbolic_U_bs = generate_symbolic_state_or_rho(U_bs, Fk_dim, precision, N_k, 2, mode_type="polarized+spectural+spatial")
    symbolic_rho_spdc_after_bs = generate_symbolic_state_or_rho(rho_spdc_after_bs, Fk_dim, precision, N_k, 2, mode_type="polarized+spectural+spatial")
    symbolic_rho_spdc_after_bs_after_dt_full_mode_final_state = generate_symbolic_state_or_rho(rho_spdc_after_bs_after_dt_full_mode_final_state, Fk_dim, precision, N_k, 2, mode_type="polarized+spectural+spatial")
    symbolic_rho_spdc_after_bs_idler = generate_symbolic_state_or_rho(rho_spdc_after_bs_idler, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural+idler")
    symbolic_rho_spdc_after_bs_signal = generate_symbolic_state_or_rho(rho_spdc_after_bs_signal, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural+idler")
    symbolic_rho_spdc_after_bs_after_dt_idler_final_state = generate_symbolic_state_or_rho(rho_spdc_after_bs_after_dt_idler_final_state, Fk_dim, precision, N_k, 0, mode_type="polarized+spectural+idler")




    latex_expr_U_spdc_matrix = (f"U_{{\\text{{spdc}}}} ="
                              + sp.latex(U_spdc_matrix) + "= "
                              + (symbolic_U_spdc.strip() if symbolic_U_spdc is not None else ""))
    latex_expr_psi_out_spdc_matrix = (f"\\left|\\Psi^{{\\text{{spdc}}}}\\right\\rangle ="
                              + sp.latex(psi_out_spdc_matrix) + "="
                              + (symbolic_psi_out_spdc.strip() if symbolic_psi_out_spdc is not None else ""))
    latex_expr_rho_out_spdc_matrix = (f"\\rho^{{\\text{{spdc}}}} ="
                              + sp.latex(rho_out_spdc_matrix) + "= "
                              + (symbolic_rho_out_spdc.strip() if symbolic_rho_out_spdc is not None else ""))
    latex_expr_rho_spdc_after_dt_final_state_matrix = (f"\\rho^{{\\text{{spdc after dt final state}}}} ="
                              + sp.latex(rho_spdc_after_dt_final_state_matrix) + "= "
                              + (symbolic_rho_spdc_after_dt_final_state.strip() if symbolic_rho_spdc_after_dt_final_state is not None else ""))
    latex_expr_U_bs_matrix = (f"U_{{\\text{{BS}}}} ="
                              + sp.latex(U_bs_matrix) + "= "
                              + (symbolic_U_bs.strip() if symbolic_U_bs is not None else ""))
    latex_expr_rho_spdc_after_bs_matrix = (f"\\rho^{{\\text{{spdc after BS}}}} ="
                              + sp.latex(rho_spdc_after_bs_matrix) + "= "
                              + (symbolic_rho_spdc_after_bs.strip() if symbolic_rho_spdc_after_bs is not None else ""))
    latex_expr_rho_spdc_after_bs_after_dt_full_mode_final_state_matrix = (f"\\rho^{{\\text{{spdc after BS after dt full mode final state}}}} ="
                              + sp.latex(rho_spdc_after_bs_after_dt_full_mode_final_state_matrix) + "= "
                              + (symbolic_rho_spdc_after_bs_after_dt_full_mode_final_state if symbolic_rho_spdc_after_bs_after_dt_full_mode_final_state is not None else ""))

    latex_expr_rho_spdc_after_bs_idler_matrix = (f"\\rho^{{\\text{{spdc after BS: idler}}}} ="
                              + sp.latex(rho_spdc_after_bs_idler_matrix) + "= "
                              + (symbolic_rho_spdc_after_bs_idler.strip() if symbolic_rho_spdc_after_bs_idler is not None else ""))
    latex_expr_rho_spdc_after_bs_signal_matrix = (f"\\rho^{{\\text{{spdc after BS: signal}}}} ="
                              + sp.latex(rho_spdc_after_bs_signal_matrix) + "= "
                              + (symbolic_rho_spdc_after_bs_signal.strip() if symbolic_rho_spdc_after_bs_signal is not None else ""))
    latex_expr_rho_spdc_after_bs_after_dt_idler_final_state_matrix = (f"\\rho^{{\\text{{spdc after BS after dt idler: final state}}}} ="
                              + sp.latex(rho_spdc_after_bs_after_dt_idler_final_state_matrix) + "= "
                              + (symbolic_rho_spdc_after_bs_after_dt_idler_final_state.strip() if symbolic_rho_spdc_after_bs_after_dt_idler_final_state is not None else ""))



    results['Fk_dim'] = Fk_dim
    results['g'] = g
    results['N_k'] = N_k
    results['T_dt_for_spdc'] = T_dt_for_spdc
    results['nth_dt_for_spdc'] = nth_dt_for_spdc
    results['phi_dt_for_spdc'] = phi_dt_for_spdc
    results['precision'] = precision
    results['total_modes'] = total_modes
    results['total_dim'] = Fk_dim ** total_modes

    results['purities_spdc_after_dt'] = purities_spdc_after_dt
    results['fidelities_spdc_after_dt'] = fidelities_spdc_after_dt

    results['P_n'] = P_n
    results['mean'] = mean
    results['std'] = std
    results['distribution'] = distribution
    results['g2_ii'] = g2_ii
    results['g2_ss'] = g2_ss
    results['g2_si'] = g2_si
    results['R_cs'] = R_cs
    results['entropy_vn_spdc'] = entropy_vn_spdc


    results['T_bs'] = T_bs
    results['nth_bs'] = nth_bs
    results['phi_bs'] = phi_bs
    results['T_dt_for_bs'] = T_dt_for_bs
    results['nth_dt_for_bs'] = nth_dt_for_bs
    results['phi_dt_for_bs'] = phi_dt_for_bs
    results['T_dt_for_bs_full_mode'] = T_dt_for_bs_full_mode
    results['nth_dt_for_bs_full_mode'] = nth_dt_for_bs_full_mode
    results['phi_dt_for_bs_full_mode'] = phi_dt_for_bs_full_mode


    results['purity_spdc_after_bs'] = purity_spdc_after_bs
    results['fidelity_spdc_after_bs'] = fidelity_spdc_after_bs
    results['purities_spdc_after_bs_after_dt_full_mode'] = purities_spdc_after_bs_after_dt_full_mode
    results['fidelities_spdc_after_bs_after_dt_full_mode'] = fidelities_spdc_after_bs_after_dt_full_mode
    results['purity_spdc_after_bs_idler'] = purity_spdc_after_bs_idler
    results['fidelity_spdc_after_bs_idler'] = fidelity_spdc_after_bs_idler
    results['purity_spdc_after_bs_signal'] = purity_spdc_after_bs_signal
    results['fidelity_spdc_after_bs_signal'] = fidelity_spdc_after_bs_signal
    results['purities_spdc_after_bs_after_dt_idler'] = purities_spdc_after_bs_after_dt_idler
    results['fidelities_spdc_after_bs_after_dt_idler'] = fidelities_spdc_after_bs_after_dt_idler


    results['Check_normalization_rho'] = rho_out_spdc
    results['H_spdc'] =  H_spdc
    results['U_spdc'] =  U_spdc
    results['psi_out_spdc'] =  psi_out_spdc #qt.Qobj(np.abs(psi_out_spdc.full()))
    results['norm_psi_out_spdc'] = psi_out_spdc.norm()
    results['rho_out_spdc'] =  rho_out_spdc #qt.Qobj(np.abs(rho_out_spdc.full()))
    results['rho_spdc_after_dt_final_state'] =  rho_spdc_after_dt_final_state #qt.Qobj(np.abs(rho_spdc_after_dt_final_state.full()))
    results['U_bs'] =  U_bs
    results['rho_spdc_after_bs'] =  rho_spdc_after_bs #qt.Qobj(np.abs(rho_spdc_after_bs.full()))
    results['rho_spdc_after_bs_after_dt_full_mode_final_state'] =  rho_spdc_after_bs_after_dt_full_mode_final_state #qt.Qobj(np.abs(rho_bs_after_dt_fullmode_final_state.full()))
    results['rho_spdc_after_bs_idler'] =  rho_spdc_after_bs_idler #qt.Qobj(np.abs(rho_spdc_after_bs_idler.full()))
    results['rho_spdc_after_bs_signal'] =  rho_spdc_after_bs_signal #qt.Qobj(np.abs(rho_spdc_after_bs_signal.full()))
    results['rho_spdc_after_bs_after_dt_idler_final_state'] =  rho_spdc_after_bs_after_dt_idler_final_state #qt.Qobj(np.abs(rho_bs_after_dt_idler_final_state.full()))



    results['symbolic_U_spdc'] = Math(latex_expr_U_spdc_matrix)
    results['symbolic_psi_out_spdc'] = Math(latex_expr_psi_out_spdc_matrix)
    results['symbolic_rho_out_spdc'] = Math(latex_expr_rho_out_spdc_matrix)
    results['symbolic_rho_spdc_after_dt_final_state'] = Math(latex_expr_rho_spdc_after_dt_final_state_matrix)
    results['symbolic_U_bs'] = Math(latex_expr_U_bs_matrix)
    results['symbolic_rho_spdc_after_bs'] = Math(latex_expr_rho_spdc_after_bs_matrix)
    results['symbolic_rho_spdc_after_bs_after_dt_full_mode_final_state'] = Math(latex_expr_rho_spdc_after_bs_after_dt_full_mode_final_state_matrix)
    results['symbolic_rho_spdc_after_bs_idler'] = Math(latex_expr_rho_spdc_after_bs_idler_matrix)
    results['symbolic_rho_spdc_after_bs_signal'] = Math(latex_expr_rho_spdc_after_bs_signal_matrix)
    results['symbolic_rho_spdc_after_bs_after_dt_idler_final_state'] = Math(latex_expr_rho_spdc_after_bs_after_dt_idler_final_state_matrix)


    results['latex_expr_U_spdc_matrix'] = latex_expr_U_spdc_matrix
    results['latex_expr_psi_out_spdc_matrix'] = latex_expr_psi_out_spdc_matrix
    results['latex_expr_rho_out_spdc_matrix'] = latex_expr_rho_out_spdc_matrix
    results['latex_expr_rho_spdc_after_dt_final_state_matrix'] = latex_expr_rho_spdc_after_dt_final_state_matrix
    results['latex_expr_U_bs_matrix'] = latex_expr_U_bs_matrix
    results['latex_expr_rho_spdc_after_bs_matrix'] = latex_expr_rho_spdc_after_bs_matrix
    results['latex_expr_rho_spdc_after_bs_after_dt_full_mode_final_state_matrix'] = latex_expr_rho_spdc_after_bs_after_dt_full_mode_final_state_matrix
    results['latex_expr_rho_spdc_after_bs_idler_matrix'] = latex_expr_rho_spdc_after_bs_idler_matrix
    results['latex_expr_rho_spdc_after_bs_signal_matrix'] = latex_expr_rho_spdc_after_bs_signal_matrix
    results['latex_expr_rho_spdc_after_bs_after_dt_idler_final_state_matrix'] = latex_expr_rho_spdc_after_bs_after_dt_idler_final_state_matrix


    return results

## 6. Example finite-Fock state and diagnostic plots

This cell runs one representative $N_k=0$, $d=2$, $g=0.3$ example and prints intermediate state metrics. It also generates sparse density-matrix and phase-space visualizations.

For the ideal TMSV pair,

$$
|\psi_g\rangle
=
\operatorname{sech}(g)
\sum_{n=0}^{\infty}
(\tanh g)^n|n,n\rangle,
$$

so the mean marginal occupation is

$$
\bar n_0=\sinh^2 g.
$$

The example is primarily diagnostic; the paper-quality state-level benchmark is produced in the later simulation-versus-theory cell.

In [ ]:
if __name__ == '__main__':
    
    cmap_name = 'PiYG' #'PiYG'
    os.makedirs("images", exist_ok=True)
    
    Fk_dim = 2
    g_values = [0.3]
    N_k_values = [0]

    # === PART 1: after SPDC ===
    T_dt_for_spdc_values = [0.80]
    nth_dt_for_spdc_values = [0.15]
    phi_dt_for_spdc = 0 * np.pi / 2

    # === PART 2: BS ===
    T_bs = 0.5
    nth_bs = 0
    phi_bs = 0 * np.pi / 2

    # === PART 3: after BS full mode ===
    T_dt_for_bs_values_full_mode = [1]
    nth_dt_for_bs_values_full_mode = [0.0]
    phi_dt_for_bs_full_mode = 0 * np.pi / 2

    # === PART 4: after BS idler/signal ===
    T_dt_for_bs_values = [1]
    nth_dt_for_bs_values = [0.0]
    phi_dt_for_bs = 0 * np.pi / 2

    precision = compute_precision(Fk_dim, g_values[0], N_k_values[0])
    params = [
        (Fk_dim, N_k, g, precision,
         T_dt_for_spdc, nth_dt_for_spdc, phi_dt_for_spdc,
         T_bs, phi_bs, nth_bs,
         T_dt_for_bs_full_mode, nth_dt_for_bs_full_mode, phi_dt_for_bs_full_mode,
         T_dt_for_bs, nth_dt_for_bs, phi_dt_for_bs)
        for g in g_values
        for N_k in N_k_values
        for T_dt_for_spdc in T_dt_for_spdc_values
        for nth_dt_for_spdc in nth_dt_for_spdc_values
        for T_dt_for_bs_full_mode in T_dt_for_bs_values_full_mode
        for nth_dt_for_bs_full_mode in nth_dt_for_bs_values_full_mode
        for T_dt_for_bs in T_dt_for_bs_values
        for nth_dt_for_bs in nth_dt_for_bs_values
    ]

    n_jobs = max(1, mp.cpu_count() - 1)

    print("\n" "============================================================\n"
        "Starting parallel SPDC calculations\n"
        f"Number of parameter sets : {len(params)}\n"
        f"Number of workers        : {n_jobs}\n"
        "Backend                  : joblib / loky\n"
        "============================================================"
         )

    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=0)(delayed(state_evolution_spdc)(p) for p in params)
    for res in results:
        if res:
            print( "\n" "*------------------------- Parameters ------------------------------*\n"
                f"Fk_dim = {res['Fk_dim']}\n"
                f"g = {res['g']:.2f}\n"
                f"N_k = {res['N_k']}\n"
                f"T_dt_for_spdc = {res['T_dt_for_spdc']:.2f}\n"
                f"nth_dt_for_spdc = {res['nth_dt_for_spdc']:.2f}\n"
                f"phi_dt_for_spdc = {res['phi_dt_for_spdc']:.2f}\n"
                f"T_bs = {res['T_bs']:.2f}\n"
                f"nth_bs = {res['nth_bs']:.2f}\n"
                f"phi_bs = {res['phi_bs']:.2f}\n"
                f"T_dt_for_bs = {res['T_dt_for_bs']:.2f}\n"
                f"nth_dt_for_bs = {res['nth_dt_for_bs']:.2f}\n"
                f"phi_dt_for_bs = {res['phi_dt_for_bs']:.2f}\n"
                f"Total modes = {res['total_modes']}\n"
                f"Total dimension = {res['total_dim']}")

            purities_spdc_after_dt = res['purities_spdc_after_dt']
            fidelities_spdc_after_dt = res['fidelities_spdc_after_dt']
            print("\n" "*------------------ Purity and Fidelity SPDC after DT --------------------*\n"
                + "\n".join(f"After mode {i}: Purity = {P:.3f}, Fidelity = {F:.3f}"
                    for i, (P, F) in enumerate(zip(purities_spdc_after_dt, fidelities_spdc_after_dt))))

            purity_spdc_after_bs = res['purity_spdc_after_bs']
            fidelity_spdc_after_bs = res['fidelity_spdc_after_bs']
            purities_spdc_after_bs_after_dt_full_mode = res['purities_spdc_after_bs_after_dt_full_mode']
            fidelities_spdc_after_bs_after_dt_full_mode = res['fidelities_spdc_after_bs_after_dt_full_mode']
            print("\n" "*----------- Purity and Fidelity after BS and after DT (full mode) -------*\n"
                f"Purity after BS: {purity_spdc_after_bs:.3f}, Fidelity after BS: {fidelity_spdc_after_bs:.3f}\n"
                + "\n".join(f"After mode {i}: Purity = {P_bs:.3f}, Fidelity = {F_bs:.3f}"
                    for i, (P_bs, F_bs) in enumerate(zip(
                        purities_spdc_after_bs_after_dt_full_mode,
                        fidelities_spdc_after_bs_after_dt_full_mode))))

            purity_spdc_after_bs_idler = res['purity_spdc_after_bs_idler']
            fidelity_spdc_after_bs_idler = res['fidelity_spdc_after_bs_idler']
            purity_spdc_after_bs_signal = res['purity_spdc_after_bs_signal']
            fidelity_spdc_after_bs_signal = res['fidelity_spdc_after_bs_signal']
            purities_spdc_after_bs_after_dt_idler = res['purities_spdc_after_bs_after_dt_idler']
            fidelities_spdc_after_bs_after_dt_idler = res['fidelities_spdc_after_bs_after_dt_idler']
            print("\n" "*--------------- Purity and Fidelity idler/signal after DT -------------*\n"
                f"Purity after BS idler: {purity_spdc_after_bs_idler:.3f}, Fidelity after BS idler: {fidelity_spdc_after_bs_idler:.3f}\n"
                f"Purity after BS signal: {purity_spdc_after_bs_signal:.3f}, Fidelity after BS signal: {fidelity_spdc_after_bs_signal:.3f}\n"
                + "\n".join(f"After mode {i}: Purity = {P_bs:.3f}, Fidelity = {F_bs:.3f}"
                    for i, (P_bs, F_bs) in enumerate(zip(
                        purities_spdc_after_bs_after_dt_idler,
                        fidelities_spdc_after_bs_after_dt_idler))))

            mean = res['mean']
            std = res['std']
            distribution = res['distribution']
            g2_ii = res['g2_ii']
            g2_si = res['g2_si']
            R_cs = res['R_cs']
            entropy_vn_spdc = res['entropy_vn_spdc']
            print("\n" "*------------ Photon Statistics SPDC (last mode) --------------------*\n"
                + "\n".join(f"{n} photon pairs → P = {p:.3f}" for n, p in sorted(res['P_n'].items()))
                + "\n"
                f"Last Mode: Mean = {mean:.3f}, Std = {std:.3f}\n"
                f"Last Mode: Distribution = {distribution}\n"
                f"Last Mode: g2_ii = {g2_ii:.3f}\n"
                f"Last Mode: g2_si = {g2_si:.3f}\n"
                f"Cauchy–Schwarz violation = {R_cs:.3f}\n"
                f"Last Mode: von Neumann entropy = {entropy_vn_spdc:.3f}"
            )

            rho = res['Check_normalization_rho']
            trace_rho = rho.tr()
            normalization_status = ("The density matrix is properly normalized."
                if abs(trace_rho - 1) < 1e-10
                else "The density matrix is NOT normalized.")
            print("\n" "*---------------- Check normalization -----------------------------*\n"
                f"Trace of rho: {trace_rho:.3f}\n" 
                  f"{normalization_status}")

            print("\n" "*------------------------ Display after SPDC ----------------------*")
            display(res['symbolic_U_spdc'])
            print(f"Is U_spdc unitary? {res['U_spdc'].isunitary}")
            display(res['symbolic_psi_out_spdc'])
            print(f"Norm of psi_out_spdc: {res['norm_psi_out_spdc']:.2f}")
            display(res['symbolic_rho_out_spdc'])
            rho_out_spdc = res["rho_out_spdc"]
            print(f"rho_out_spdc: {rho_out_spdc.dims}")
            plot_sparse_density_matrix(rho_out_spdc,total_modes=2 + 4 * res['N_k'],Fk_dim=Fk_dim,N_k=res['N_k'], threshold=1e-4,cmap_name=cmap_name)
            display(res['symbolic_rho_spdc_after_dt_final_state'])
            rho_spdc_after_dt_final_state = res["rho_spdc_after_dt_final_state"]
            plot_sparse_density_matrix(rho_spdc_after_dt_final_state,total_modes=2 + 4 * res['N_k'],Fk_dim=Fk_dim,N_k=res['N_k'],threshold=1e-4,cmap_name=cmap_name, 
                 save_path=(f"images/rho_spdc_after_dt_final_state_" f"g{res['g']:.2f}_Nk{res['N_k']}_T{res['T_dt_for_spdc']:.2f}_nth{res['nth_dt_for_spdc']:.2f}.png"))
            plot_wigner(rho_spdc_after_dt_final_state, Fk_dim,
                save_path=(f"images/wigner_after_dt_final_state_" f"g{res['g']:.2f}_Nk{res['N_k']}.png"))

            print("\n" "*--------------------------- Display after BS ----------------------------*")
            display(res['symbolic_U_bs'])
            print(f"Is U_bs unitary? {res['U_bs'].isunitary}")
            display(res['symbolic_rho_spdc_after_bs'])
            rho_spdc_after_bs = res["rho_spdc_after_bs"]
            print(f"rho_spdc_after_bs: {rho_spdc_after_bs.dims}")
            plot_sparse_density_matrix(rho_spdc_after_bs,total_modes=2 + 4 * res['N_k'] + 2, Fk_dim=Fk_dim, N_k=res['N_k'], threshold=1e-4, cmap_name=cmap_name)
            display(res['symbolic_rho_spdc_after_bs_after_dt_full_mode_final_state'])
            rho_spdc_after_bs_after_dt_full_mode_final_state = res["rho_spdc_after_bs_after_dt_full_mode_final_state"]
            print(f"rho_spdc_after_bs_after_dt_full_mode_final_state: "f"{rho_spdc_after_bs_after_dt_full_mode_final_state.dims}")
            plot_sparse_density_matrix(rho_spdc_after_bs_after_dt_full_mode_final_state,total_modes=2 + 4 * res['N_k'] + 2,Fk_dim=Fk_dim,N_k=res['N_k'],threshold=1e-4, cmap_name=cmap_name)
            plot_wigner(rho_spdc_after_bs_after_dt_full_mode_final_state, Fk_dim)

            print("\n" "*-------------------------- Display idler/signal ------------------------*")
            display(res['symbolic_rho_spdc_after_bs_idler'])
            rho_spdc_after_bs_idler = res["rho_spdc_after_bs_idler"]
            plot_sparse_density_matrix(rho_spdc_after_bs_idler,total_modes=2 + 4 * res['N_k'],Fk_dim=Fk_dim,N_k=res['N_k'],threshold=1e-4, cmap_name=cmap_name)
            plot_wigner(rho_spdc_after_bs_idler, Fk_dim)
            display(res['symbolic_rho_spdc_after_bs_after_dt_idler_final_state'])
            rho_spdc_after_bs_after_dt_idler_final_state = res["rho_spdc_after_bs_after_dt_idler_final_state"]
            print(f"rho_spdc_after_bs_after_dt_idler_final_state: " f"{rho_spdc_after_bs_after_dt_idler_final_state.dims}")
            plot_sparse_density_matrix(rho_spdc_after_bs_after_dt_idler_final_state, total_modes=2 + 4 * res['N_k'],Fk_dim=Fk_dim,N_k=res['N_k'],threshold=1e-4, cmap_name=cmap_name)
            plot_wigner(rho_spdc_after_bs_after_dt_idler_final_state, Fk_dim)
            display(res['symbolic_rho_spdc_after_bs_signal'])
            rho_spdc_after_bs_signal = res["rho_spdc_after_bs_signal"]
            plot_sparse_density_matrix(rho_spdc_after_bs_signal,total_modes=2 + 4 * res['N_k'],Fk_dim=Fk_dim,N_k=res['N_k'],threshold=1e-4, cmap_name=cmap_name)
            plot_wigner(rho_spdc_after_bs_signal, Fk_dim)



            

## 7. Exact Gaussian purity and fidelity

For one TMSV pair propagated through identical thermal-loss channels, define

$$
c=\cosh(2g),\qquad s=\sinh(2g),
$$

$$
A=T c+(1-T)(2\bar n_{\rm th}+1),
\qquad
C=T s.
$$

The output covariance matrix is

$$
V_{\rm out}
=
\frac12
\begin{pmatrix}
A I_2 & C Z\\
C Z & A I_2
\end{pmatrix},
\qquad
Z=\operatorname{diag}(1,-1).
$$

The exact single-pair state metrics are

$$
\mathcal P_{\rm pair}=\frac{1}{A^2-C^2},
$$

$$
\mathcal F_{\rm pair}
=
\frac{4}{(c+A)^2-(s+C)^2}.
$$

Because the digital-twin Hamiltonian factorizes into $M=1+2N_k$ independent pairs,

$$
\mathcal P_{N_k}=\mathcal P_{\rm pair}^{M},
\qquad
\mathcal F_{N_k}=\mathcal F_{\rm pair}^{M}.
$$

This cell produces the analytical four-panel gain/transmissivity dependence.

In [ ]:
# Theory 1

def purity_fidelity_theory(g, T, nth, N_k):
    g, T = np.asarray(g), np.asarray(T)
    M = 1 + 2*N_k
    c, s = np.cosh(2*g), np.sinh(2*g)
    A = T*c + (1-T)*(2*nth+1)
    C = T*s
    P = (1/(A**2-C**2))**M
    F = (4/((c+A)**2-(s+C)**2))**M
    return P, F

def format_ax(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel, fontsize=15)
    ax.set_ylabel(ylabel, fontsize=15)
    ax.xaxis.set_minor_locator(AutoMinorLocator(4))
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax.tick_params(which='major', direction='in', length=6, width=1,
                   top=True, right=True, labelsize=11)
    ax.tick_params(which='minor', direction='in', length=3, width=0.8,
                   top=True, right=True)
    ax.grid(which='major', ls='--', lw=0.7, alpha=0.5)
    ax.grid(which='minor', ls='--', lw=0.4, alpha=0.25)
    ax.legend(frameon=True, fontsize=10, facecolor="#F0F2F1",
              edgecolor="none", loc="best")

N_k_values = [0, 1, 5]
g0 = 0.3
g_values = np.linspace(0, 1, 300)
T_values = np.linspace(0, 1, 300)

cases = [
    ("Ideal",    1.00, 0.00, '-'),
    ("Nonideal", 0.85, 0.03, '--')
]

cmap = plt.colormaps["tab10"]



fig, axes = plt.subplots(2, 2, figsize=(7, 7.6))

labels = ['(a)', '(b)', '(c)', '(d)']
for ax, label in zip(axes.flat, labels):
    ax.text(-0.13, 1.05, label,
            transform=ax.transAxes,
            fontsize=16, fontweight='bold',
            va='bottom', ha='left',
            clip_on=False)

# (a) Purity vs g
ax = axes[0,0]
for i, N_k in enumerate(N_k_values):
    color = cmap(i)
    for _, T, nth, ls in cases:
        P, _ = purity_fidelity_theory(g_values, T, nth, N_k)
        P0, _ = purity_fidelity_theory(g0, T, nth, N_k)
        label = fr"$N_k={N_k},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$"
        ax.plot(g_values, P, ls=ls, color=color, lw=1.5, label=label)
        # ax.plot(g0, P0, 'o', mfc='none', mec=color, ms=6)
# ax.axvline(g0, color='k', ls=':', lw=1)
ax.set_xlabel(r"SPDC gain ($g$)", fontsize=15)
ax.set_ylabel("Purity", fontsize=15)

# (b) Fidelity vs g
ax = axes[0,1]
for i, N_k in enumerate(N_k_values):
    color = cmap(i)
    for _, T, nth, ls in cases:
        _, F = purity_fidelity_theory(g_values, T, nth, N_k)
        _, F0 = purity_fidelity_theory(g0, T, nth, N_k)
        label = fr"$N_k={N_k},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$"
        ax.plot(g_values, F, ls=ls, color=color, lw=1.5, label=label)
        #ax.plot(g0, F0, 'o', mfc='none', mec=color, ms=6)
#ax.axvline(g0, color='k', ls=':', lw=1)
ax.set_xlabel(r"SPDC gain ($g$)", fontsize=15)
ax.set_ylabel("Fidelity", fontsize=15)

# (c) Purity vs T, g = 0.06
ax = axes[1,0]
for i, N_k in enumerate(N_k_values):
    color = cmap(i)
    for nth, T_mark, ls in [(0.00, 1.00, '-'), (0.03, 0.85, '--')]:
        P, _ = purity_fidelity_theory(g0, T_values, nth, N_k)
        P0, _ = purity_fidelity_theory(g0, T_mark, nth, N_k)
        label = fr"$N_k={N_k},\,\bar{{n}}_{{th}}={nth:.2f}$"
        ax.plot(T_values, P, ls=ls, color=color, lw=1.5, label=label)
        # ax.plot(T_mark, P0, 'o', mfc='none', mec=color, ms=6)
# ax.axvline(0.85, color='k', ls=':', lw=1)
ax.set_xlabel(r"Transmission ($T$)", fontsize=15)
ax.set_ylabel("Purity", fontsize=15)

# (d) Fidelity vs T, g = 0.06
ax = axes[1,1]
for i, N_k in enumerate(N_k_values):
    color = cmap(i)
    for nth, T_mark, ls in [(0.00, 1.00, '-'), (0.03, 0.85, '--')]:
        _, F = purity_fidelity_theory(g0, T_values, nth, N_k)
        _, F0 = purity_fidelity_theory(g0, T_mark, nth, N_k)
        label = fr"$N_k={N_k},\,\bar{{n}}_{{th}}={nth:.2f}$"
        ax.plot(T_values, F, ls=ls, color=color, lw=1.5, label=label)
        # ax.plot(T_mark, F0, 'o', mfc='none', mec=color, ms=6)
# ax.axvline(0.85, color='k', ls=':', lw=1)
ax.set_xlabel(r"Transmission ($T$)", fontsize=15)
ax.set_ylabel("Fidelity", fontsize=15)

for ax in axes.flat:
    ax.set_ylim(0, 1.05)
    ax.xaxis.set_minor_locator(AutoMinorLocator(4))
    ax.yaxis.set_minor_locator(AutoMinorLocator(4))
    ax.tick_params(which='major', direction='in', length=6,top=True, right=True, labelsize=11)
    ax.tick_params(which='minor', direction='in', length=3, top=True, right=True)
    ax.grid(which='major', ls='--', lw=0.7, alpha=0.5)
    ax.grid(which='minor', ls='--', lw=0.4, alpha=0.25)
    ax.legend(fontsize=10, frameon=True, facecolor="#FFFDE8", edgecolor="none")

axes[0,0].set_xlim(0, 1)
axes[0,1].set_xlim(0, 1)
axes[1,0].set_xlim(0, 1)
axes[1,1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig("images/theory_purity_fidelity_4panels.png", dpi=300, bbox_inches="tight")
plt.show()

## 8. Ternary visualization of analytical state metrics

The same closed-form purity and fidelity are evaluated on a constrained simplex,

$$
\tau_N+\tau_T+\tau_{\rm th}=1,
$$

with the visualization map

$$
N_k=N_{k,\max}\tau_N,\qquad
T=\tau_T,\qquad
\bar n_{\rm th}=\bar n_{{\rm th},\max}\tau_{\rm th}.
$$

This is a **two-dimensional slice**, not a projection of the full three-parameter volume. Noninteger $N_k$ values in this plot are interpolation coordinates only.

The cell exports both the plotted fields and contour paths to CSV for independent figure recreation.

In [ ]:
# Theory 2 
def ternary_pin(ax, point, text_pos, text, fontsize=12):
    t, l, r = point
    tt, ll, rr = text_pos
    # pin line
    ax.plot([t, tt], [l, ll], [r, rr],color='white', lw=0.8, zorder=8)
    # hollow marker
    ax.plot(t, l, r, 'o',ms=4, mfc='white', mec='k', mew=0.9,zorder=10)
    # label
    ax.text(tt, ll, rr, text,fontsize=fontsize,color='white',ha='left', va='center',zorder=10)

def physical_to_ternary(Nk, T, nth):
    a = Nk/10
    b = T
    c = nth/2
    s = a + b + c
    return a/s, b/s, c/s

    
def purity_fidelity_theory(g, T, nth, N_k):
    g, T = np.asarray(g), np.asarray(T)
    M = 1 + 2*N_k
    c, s = np.cosh(2*g), np.sinh(2*g)
    A = T*c + (1-T)*(2*nth+1)
    C = T*s
    P = (1/(A**2-C**2))**M
    F = (4/((c+A)**2-(s+C)**2))**M
    return P, F

def ternary_grid(n=101):
    nk, tt, nn = [], [], []
    for i in range(n):
        a = i/(n-1)
        for j in range(n-i):
            b = j/(n-1)
            c = 1-a-b
            nk.append(a)
            tt.append(b)
            nn.append(c)
    return np.array(nk), np.array(tt), np.array(nn)

# Fixed SPDC gain
g_fixed = 0.3
# Normalized ternary coordinates
xNk, xT, xn = ternary_grid(121)
# Convert back to physical variables
N_k = 10*xNk          # 0 --> 10
T = xT                # 0 --> 1
nth = 2*xn            # 0 --> 2
# Theory
P, F = purity_fidelity_theory(g_fixed, T, nth, N_k)
# Numerical safety
P = np.clip(P, 0, 1)
F = np.clip(F, 0, 1)

levels = np.linspace(0, 1, 201)

fig = plt.figure(figsize=(10, 4.8))
ax1 = fig.add_subplot(1, 2, 1, projection="ternary")
ax2 = fig.add_subplot(1, 2, 2, projection="ternary")


# PURITY
cp = ax1.tricontourf(xNk, xT, xn, P,levels=levels,cmap="turbo",vmin=0, vmax=1)
# contour boundaries
contour_P = ax1.tricontour(xNk, xT, xn, P,levels=np.arange(0.1, 1.0, 0.1),colors="k",linewidths=0.25,alpha=0.35)

ternary_pin(ax1,physical_to_ternary(0.3, 0.95, 0.03),(0.08, 0.82, 0.10),"High-purity regime")
#ternary_pin(ax1,physical_to_ternary(2, 0.65, 0.15),(0.28, 0.58, 0.14),"Loss-induced\nmixing")
ternary_pin(ax1,physical_to_ternary(2, 0.35, 1.0),(0.30, 0.28, 0.52),"Noise\ndominated")
#ternary_pin(ax1,physical_to_ternary(8, 0.55, 0.20),(0.70, 0.22, 0.08),"Multimode\npenalty")


# FIDELITY
cf = ax2.tricontourf(xNk, xT, xn, F,levels=levels,cmap="hot",vmin=0, vmax=1)
contour_F = ax2.tricontour(xNk, xT, xn, F,levels=np.arange(0.1, 1.0, 0.1),colors="k",linewidths=0.25,alpha=0.35)

#ternary_pin(ax2,physical_to_ternary(0.0, 0.95, 0.03),(0.08, 0.82, 0.10),"High-fidelity regime")
ternary_pin(ax2,physical_to_ternary(2, 0.65, 0.15),(0.28, 0.58, 0.14),"Loss-induced mixing")
#ternary_pin(ax2,physical_to_ternary(2, 0.35, 1.0),(0.30, 0.28, 0.42),"Thermal-noise limited")
ternary_pin(ax2,physical_to_ternary(8, 0.55, 0.20),(0.70, 0.35, 0.08),"Multimode\npenalty")

# Axis labels and physical tick ranges
ticks = np.linspace(0, 1, 6)
for ax in (ax1, ax2):
    # Top facet: Nk = 0 --> 10
    ax.set_tlabel(r"Effective multimode parameter $N_k$", fontsize=12)
    ax.taxis.set_ticks(ticks,labels=[f"{10*x:.0f}" for x in ticks])
    # Left facet: T = 0 --> 1
    ax.set_llabel(r"Transmission $T$", fontsize=12)
    ax.laxis.set_ticks(ticks,labels=[f"{x:.1f}" for x in ticks])
    # Right facet: nth = 0 --> 2
    ax.set_rlabel(r"Mean thermal photon number $\bar{n}_{\mathrm{th}}$", fontsize=12)
    ax.raxis.set_ticks(ticks,labels=[f"{2*x:.1f}" for x in ticks])
    ax.taxis.set_label_position("tick1")
    ax.laxis.set_label_position("tick1")
    ax.raxis.set_label_position("tick1")
    ax.taxis.set_ticks_position("tick1")
    ax.laxis.set_ticks_position("tick1")
    ax.raxis.set_ticks_position("tick1")

    ax.tick_params(labelsize=12, direction="in")
    #ax.grid(True,linestyle="--",linewidth=0.6,alpha=0.35)


ax1.text(-0.08, 1.03, "(a)",transform=ax1.transAxes,fontsize=16,fontweight="bold")
ax2.text(-0.08, 1.03, "(b)",transform=ax2.transAxes,fontsize=16,fontweight="bold")
# Colorbars
cbar1 = fig.colorbar(cp, ax=ax1,fraction=0.03,pad=0.10,ticks=np.arange(0, 1.01, 0.2))
cbar1.set_label("Purity", fontsize=13)
cbar2 = fig.colorbar(cf, ax=ax2,fraction=0.03,pad=0.10,ticks=np.arange(0, 1.01, 0.2))
cbar2.set_label("Fidelity", fontsize=13)

plt.tight_layout()
plt.savefig("images/ternary_purity_fidelity_contour.png",dpi=300,bbox_inches="tight")
plt.show()




#---------------------- DATA------------------------------------

os.makedirs("csv_data",exist_ok=True)
h=np.sqrt(3)/2
n=121

# Cartesian coordinates matching the LaTeX triangle
x=0.5*xNk+xn
y=h*xNk

# Index of point (i,j) in ternary_grid()
def idx(i,j):
    return i*n-i*(i-1)//2+j

# Save triangular patches directly as CSV
rows=[]
for i in range(n-1):
    for j in range(n-i-1):
        triangles=[[idx(i,j),idx(i,j+1),idx(i+1,j)]]
        if j<n-i-2:
            triangles.append([idx(i,j+1),idx(i+1,j+1),idx(i+1,j)])
        for triangle in triangles:
            for k in triangle:
                rows.append({
                    "x":x[k],"y":y[k],
                    "N_k":N_k[k],"T":T[k],"n_th":nth[k],"g":g_fixed,
                    "Purity":P[k],"Fidelity":F[k]})

pd.DataFrame(rows).to_csv("csv_data/ternary_full_data.csv",index=False)



def save_contours(contour_set,filename):
    rows=[]
    for level,segments in zip(contour_set.levels,contour_set.allsegs):
        for segment_id,segment in enumerate(segments):
            for point_id,(x,y) in enumerate(segment):
                rows.append({
                    "level":level,
                    "segment":segment_id,
                    "point":point_id,
                    "x":0.5+h*x,
                    "y":h*y
                })
            rows.append({
                "level":np.nan,
                "segment":np.nan,
                "point":np.nan,
                "x":np.nan,
                "y":np.nan
            })
    pd.DataFrame(rows).to_csv(
        filename,index=False,na_rep="nan"
    )

save_contours(contour_P,"csv_data/ternary_purity_contours.csv")
save_contours(contour_F,"csv_data/ternary_fidelity_contours.csv")

## 9. Direct state-level density-matrix validation

This cell compares the finite-Fock $N_k=0$ calculation with the cutoff-independent Gaussian formulas.

The benchmark tests:

- purity and squared Uhlmann–Jozsa fidelity versus gain $g$;
- purity and fidelity versus channel transmissivity $T$;
- several thermal occupations $\bar n_{\rm th}$.

The manuscript uses $d=6$ for the final state-level comparison after the independent convergence study in Section 18 below.

In [ ]:
# Simulation + theory
def purity_fidelity_theory(g, T, nth, N_k):
    g, T = np.asarray(g), np.asarray(T)
    M = 1 + 2*N_k
    c, s = np.cosh(2*g), np.sinh(2*g)
    A = T*c + (1-T)*(2*nth+1)
    C = T*s
    purity_pair = 1/(A**2-C**2)
    fidelity_pair = 4/((c+A)**2-(s+C)**2)
    return purity_pair**M, fidelity_pair**M

def plot_purity_fidelity_vs_g_T(Fk_dim, N_k, g_values, results_g, T_values, results_T, g_fixed=0.3):
    plt.rcParams['text.usetex'] = False
    mode = 2 + 4*N_k - 1
    cmap = plt.colormaps["tab10"]
    os.makedirs("csv_data", exist_ok=True)
    data_g, data_T = [], []

    fig, axes = plt.subplots(2, 2, figsize=(7, 7.6))
    labels = ['(a)', '(b)', '(c)', '(d)']
    for ax, label in zip(axes.flat, labels):
        ax.text(-0.13, 1.05, label, transform=ax.transAxes, fontsize=16, fontweight='bold', va='bottom', ha='left', clip_on=False)

    # ---------- vs g ----------
    groups_g = defaultdict(list)
    for r in results_g:
        if r and r['N_k'] == N_k:
            groups_g[(r['T_dt_for_spdc'], r['nth_dt_for_spdc'])].append(r)

    for i, ((T_val, nth_val), cfg) in enumerate(groups_g.items()):
        gs, P, F = [], [], []
        for g in g_values:
            r = next((r for r in cfg if np.isclose(r['g'], g)), None)
            if r is not None:
                gs.append(g)
                P_sim = np.real(r['purities_spdc_after_dt'][mode])
                F_sim = np.real(r['fidelities_spdc_after_dt'][mode])
                P.append(P_sim)
                F.append(F_sim)
                P_th, F_th = purity_fidelity_theory(g, T_val, nth_val, N_k)
                data_g.append({
                    "N_k": N_k,
                    "g": g,
                    "T": T_val,
                    "n_th": nth_val,
                    "Purity_simulation": P_sim,
                    "Purity_theory": float(P_th),
                    "Fidelity_simulation": F_sim,
                    "Fidelity_theory": float(F_th)
                })

        if gs:
            color = cmap(i % cmap.N)
            label = fr"$T={T_val:.2f},\ \bar{{n}}_{{th}}={nth_val:.2f}$"
            axes[0,0].plot(gs, P, '^', color=color, markerfacecolor='none', markeredgecolor=color, markeredgewidth=1.3, markersize=5, label=label)
            axes[0,1].plot(gs, F, 'o', color=color, markerfacecolor='none', markeredgecolor=color, markeredgewidth=1.3, markersize=5, label=label)
            P_th, F_th = purity_fidelity_theory(g_values, T_val, nth_val, N_k)
            axes[0,0].plot(g_values, P_th, 'k--', lw=1.0)
            axes[0,1].plot(g_values, F_th, 'k--', lw=1.0)

    axes[0,0].plot([], [], 'k--', lw=1.0, label="Theory")
    axes[0,1].plot([], [], 'k--', lw=1.0, label="Theory")

    df_g = pd.DataFrame(data_g)
    df_g.to_csv(f"csv_data/purity_fidelity_vs_g_Nk{N_k}.csv", index=False)

    # ---------- vs T ----------
    groups_T = defaultdict(list)
    for r in results_T:
        if r and r['N_k'] == N_k:
            groups_T[r['nth_dt_for_spdc']].append(r)

    for i, (nth_val, cfg) in enumerate(sorted(groups_T.items())):
        Ts, P, F = [], [], []
        for T in T_values:
            r = next((r for r in cfg if np.isclose(r['T_dt_for_spdc'], T) and np.isclose(r['g'], g_fixed)), None)
            if r is not None:
                Ts.append(T)
                P_sim = np.real(r['purities_spdc_after_dt'][mode])
                F_sim = np.real(r['fidelities_spdc_after_dt'][mode])
                P.append(P_sim)
                F.append(F_sim)
                P_th, F_th = purity_fidelity_theory(g_fixed, T, nth_val, N_k)
                data_T.append({
                    "N_k": N_k,
                    "g": g_fixed,
                    "T": T,
                    "n_th": nth_val,
                    "Purity_simulation": P_sim,
                    "Purity_theory": float(P_th),
                    "Fidelity_simulation": F_sim,
                    "Fidelity_theory": float(F_th)
                })

        if Ts:
            color = cmap(i % cmap.N)
            label = fr"$\bar{{n}}_{{th}}={nth_val:.2f}$"
            axes[1,0].plot(Ts, P, '^', color=color, markerfacecolor=color, markeredgecolor=color, markeredgewidth=1.3, markersize=5, label=label)
            axes[1,1].plot(Ts, F, 'o', color=color, markerfacecolor=color, markeredgecolor=color, markeredgewidth=1.3, markersize=5, label=label)
            P_th, F_th = purity_fidelity_theory(g_fixed, T_values, nth_val, N_k)
            axes[1,0].plot(T_values, P_th, 'k--', lw=1.0)
            axes[1,1].plot(T_values, F_th, 'k--', lw=1.0)

    axes[1,0].plot([], [], 'k--', lw=1.0, label="Theory")
    axes[1,1].plot([], [], 'k--', lw=1.0, label="Theory")

    df_T = pd.DataFrame(data_T)
    df_T.to_csv(f"csv_data/purity_fidelity_vs_T_Nk{N_k}.csv", index=False)

    axes[0,0].set_ylabel("Purity", fontsize=15)
    axes[0,1].set_ylabel("Fidelity", fontsize=15)
    axes[1,0].set_ylabel("Purity", fontsize=15)
    axes[1,1].set_ylabel("Fidelity", fontsize=15)

    for ax in axes[0]:
        ax.set_xlabel(r"SPDC gain ($g$)", fontsize=15)
        ax.set_xlim(-0.03, 1.03)
        ax.set_xticks(np.arange(0, 1.01, 0.2))

    for ax in axes[1]:
        ax.set_xlabel(r"Transmission ($T$)", fontsize=15)
        ax.set_xlim(-0.03, 1.03)
        ax.set_xticks(np.arange(0, 1.01, 0.2))

    for ax in axes.flat:
        ax.set_ylim(0.6, 1.05)
        ax.xaxis.set_minor_locator(AutoMinorLocator(4))
        ax.yaxis.set_minor_locator(AutoMinorLocator(4))
        ax.tick_params(axis='both', which='major', direction='in', length=6, width=1.0, top=True, right=True, labelsize=11)
        ax.tick_params(axis='both', which='minor', direction='in', length=3, width=0.8, top=True, right=True)
        ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.5)
        ax.grid(True, which='minor', linestyle='--', linewidth=0.4, alpha=0.25)
        ax.legend(frameon=True, fontsize=11, facecolor="#FFFDE8", edgecolor="none", loc="best")

    plt.tight_layout()
    plt.savefig("images/plot_purity_fidelity_vs_g_T_spdc.png", dpi=300)
    plt.show()

if __name__ == '__main__':
    Fk_dim = 2
    N_k_values = [0]
    T_bs, nth_bs, phi_bs = 0.5, 0, 0
    T_dt_for_bs_values_full_mode = [1]
    nth_dt_for_bs_values_full_mode = [0.0]
    phi_dt_for_bs_full_mode = 0
    T_dt_for_bs_values = [1]
    nth_dt_for_bs_values = [0.0]
    phi_dt_for_bs = 0

    phi_dt_for_spdc = 0

    # ---------- vs g ----------
    g_values = np.linspace(0, 1, 15)
    T_nth_pairs = [(1, 0.08), (0.9, 0.08), (0.8, 0.08)]
    precision_g = compute_precision(Fk_dim, g_values[0], N_k_values[0])
    params_g = [
        (Fk_dim, N_k, g, precision_g, T_spdc, nth_spdc, phi_dt_for_spdc, T_bs, phi_bs, nth_bs, T_full, nth_full, phi_dt_for_bs_full_mode, T_out, nth_out, phi_dt_for_bs)
        for g in g_values
        for N_k in N_k_values
        for T_spdc, nth_spdc in T_nth_pairs
        for T_full in T_dt_for_bs_values_full_mode
        for nth_full in nth_dt_for_bs_values_full_mode
        for T_out in T_dt_for_bs_values
        for nth_out in nth_dt_for_bs_values
    ]

    n_jobs = max(1, mp.cpu_count()-1)
    results_g = Parallel(n_jobs=n_jobs, backend="loky")(delayed(state_evolution_spdc)(p) for p in params_g)

    # ---------- vs T, fixed g ----------
    g_fixed = 0.3
    T_spdc_values = np.linspace(0, 1, 15)
    nth_spdc_values = [0, 0.02, 0.08]
    precision_T = compute_precision(Fk_dim, g_fixed, N_k_values[0])
    params_T = [
        (Fk_dim, N_k, g_fixed, precision_T, T_spdc, nth_spdc, phi_dt_for_spdc, T_bs, phi_bs, nth_bs, T_full, nth_full, phi_dt_for_bs_full_mode, T_out, nth_out, phi_dt_for_bs)
        for N_k in N_k_values
        for T_spdc in T_spdc_values
        for nth_spdc in nth_spdc_values
        for T_full in T_dt_for_bs_values_full_mode
        for nth_full in nth_dt_for_bs_values_full_mode
        for T_out in T_dt_for_bs_values
        for nth_out in nth_dt_for_bs_values
    ]

    results_T = Parallel(n_jobs=n_jobs, backend="loky")(delayed(state_evolution_spdc)(p) for p in params_T)

    for N_k in N_k_values:
        plot_purity_fidelity_vs_g_T(Fk_dim, N_k, g_values, results_g, T_spdc_values, results_T, g_fixed)



        

## 10. Reduced polarization-count theory

The measurement-level analytical model treats the post-NPBS signal as a weak-gain correlated pair contribution plus an angle-independent thermal background.

For

$$
\bar n_0=\sinh^2 g,\qquad
b=(1-T)\bar n_{\rm th},
$$

define

$$
Q=T^2\eta(1-\eta)\bar n_0,
\qquad
B_{\rm th}=M^2b(T\bar n_0+b).
$$

The four polarization-resolved coincidence moments are

$$
C_{HH}=C_{VV}=B_{\rm th}+MQ\sin^2(\alpha-\beta),
$$

$$
C_{HV}=C_{VH}=B_{\rm th}+MQ\cos^2(\alpha-\beta).
$$

Therefore,

$$
V_{\rm pol}=\frac{MQ}{MQ+2B_{\rm th}},
\qquad
S_{\rm CHSH}^{(1)}=2\sqrt2\,V_{\rm pol}.
$$

This cell generates the reduced-theory singles, coincidence fringes, visibility, and conditional-CHSH curves.

In [ ]:
def pol_theory_parameters(g,T,nth,N_k,eta_bs=0.5):
    g,T=np.asarray(g),np.asarray(T)
    M=1+2*N_k
    n0=np.sinh(g)**2
    b=(1-T)*nth
    Q=T**2*eta_bs*(1-eta_bs)*n0
    Bth=M**2*b*(T*n0+b)
    return M,Q,Bth

def singles_theory(g,T,nth,N_k,eta_bs=0.5):
    M=1+2*N_k
    n0=np.sinh(g)**2
    ni=T*eta_bs*n0+(1-T)*nth
    ns=T*(1-eta_bs)*n0+(1-T)*nth
    return {'idler':M*ni,'signal':M*ns}

def coincidence_theory(alpha,beta,g,T,nth,N_k,eta_bs=0.5):
    M,Q,Bth=pol_theory_parameters(g,T,nth,N_k,eta_bs)
    d=np.deg2rad(np.asarray(alpha)-beta)
    S=M*Q
    same=Bth+S*np.sin(d)**2
    cross=Bth+S*np.cos(d)**2
    return {'HH':same,'HV':cross,'VH':cross,'VV':same}

def visibility_theory(g,T,nth,N_k,eta_bs=0.5):
    M,Q,Bth=pol_theory_parameters(g,T,nth,N_k,eta_bs)
    S=M*Q
    with np.errstate(divide='ignore',invalid='ignore'):
        V=S/(S+2*Bth)
    return np.where(S+2*Bth>1e-15,V,np.nan)

def correlator_theory(alpha,beta,g,T,nth,N_k,eta_bs=0.5):
    V=visibility_theory(g,T,nth,N_k,eta_bs)
    return -V*np.cos(2*np.deg2rad(np.asarray(alpha)-beta))

def chsh_theory(g,T,nth,N_k,eta_bs=0.5):
    return 2*np.sqrt(2)*visibility_theory(g,T,nth,N_k,eta_bs)



g,eta_bs=0.04,0.5
theta=np.linspace(0,361,361)
beta=0.0
N_measurements=1e6
cmap=plt.colormaps["tab10"]

fig,axes=plt.subplots(2,2,figsize=(15.3,8.5))
for ax,lab in zip(axes.flat,['(a)','(b)','(c)','(d)']):
    ax.text(-0.13,1.05,lab,transform=ax.transAxes,fontsize=16, fontweight='bold',va='bottom',ha='left',clip_on=False)


cases=[
    (0,1.00,0.00),
    (0,0.85,0.1),
    (1,1.00,0.00),
    (1,0.85,0.1)
]

# (a) Singles vs analyzer angle
ax=axes[0,0]
for i,(Nk,T,nth) in enumerate(cases):
    S=singles_theory(g,T,nth,Nk,eta_bs)['idler']
    ax.plot(theta,np.full_like(theta,N_measurements*S),
            lw=1.5,color=cmap(i),
            label=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$")

ax.set(xlabel=r"Idler analyzer angle $\alpha$ [deg.]",
       ylabel="Expected singles counts",xlim=(0,360))
ax.set_xticks(np.arange(0,361,30))

# (b) Coincidences vs analyzer angle
ax=axes[0,1]
for i,(Nk,T,nth) in enumerate(cases):
    C=coincidence_theory(theta,beta,g,T,nth,Nk,eta_bs)
    label=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$"

    ax.plot(theta,N_measurements*C['HH'],'-',lw=1.5,color=cmap(i),label=label+r", $HH$")
    ax.plot(theta,N_measurements*C['HV'],'--',lw=1.5,color=cmap(i),label=label+r", $HV$")

ax.set(xlabel=r"Idler analyzer angle $\alpha$ [deg.]",ylabel="Expected coincidence counts",xlim=(0,360))
ax.set_xticks(np.arange(0,361,30))



Nk_values=[0,1,4,10]
nth_values=[0.00,0.05]
T_values=np.linspace(1e-4,1,1000)

# (c) Visibility vs T
ax=axes[1,0]
for i,Nk in enumerate(Nk_values):
    for nth in nth_values:
        V=visibility_theory(g,T_values,nth,Nk,eta_bs)
        ax.plot(T_values,V,'-' if nth==0 else '--',lw=1.5,color=cmap(i), label=fr"$N_k={Nk},\ \bar{{n}}_{{th}}={nth:.2f}$")

ax.axhline(1/np.sqrt(2),color='k',ls=':',lw=1.3,label=r"$V=1/\sqrt{2}$")
ax.set(xlabel=r"Transmissivity $T$",ylabel=r"Polarization visibility $V_{\rm pol}$",xlim=(0,1),ylim=(0,1.05))
ax.set_xticks(np.arange(0,1.01,0.2))

# (d) CHSH vs T
ax=axes[1,1]
for i,Nk in enumerate(Nk_values):
    for nth in nth_values:
        S=chsh_theory(g,T_values,nth,Nk,eta_bs)
        ax.plot(T_values,S,'-',lw=1.5, label=fr"$N_k={Nk},\ \bar{{n}}_{{th}}={nth:.2f}$")

ax.set_xticks(np.arange(0,1.01,0.2))
ax.axhline(2,color='k',ls='--',lw=1.2,label=r"Local bound $S=2$")
ax.axhline(2*np.sqrt(2),color='k',ls=':',lw=1.2,label=r"Quantum bound $S=2\sqrt{2}$")
ax.set(xlabel=r"Transmissivity $T$",ylabel=r"CHSH parameter $S_{\rm CHSH}$",xlim=(0,1),ylim=(0,3))
ax.axhspan(0,2.0,facecolor="lightblue",alpha=0.3,label="CHSH-nonviolating regime")
ax.axhspan(2.0,2.8284,facecolor="lightgreen",alpha=0.25,label="CHSH-violating regime")
ax.text(0.06,1.0,"CHSH-nonviolating\nregime",fontsize=14,color="blue",alpha=0.8)
ax.text(0.06,2.25,"CHSH-violating\nregime",fontsize=14,color="green",alpha=0.8)


for ax in axes.flat:
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(which='major',direction='in',length=6,top=True,right=True,labelsize=13)
    ax.tick_params(which='minor',direction='in',length=3,top=True,right=True)
    ax.grid(which='major',ls='--',lw=0.6,alpha=0.4)
    ax.grid(which='minor',ls='--',lw=0.35,alpha=0.2)
    ax.legend(fontsize=12,frameon=True,facecolor="#F5F6FF",edgecolor="none",bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.savefig("images/polarization_theory_4panels.png",dpi=300,bbox_inches="tight")
plt.show()

## 11. Polarization measurement operators and finite-Fock simulation

The post-NPBS state contains idler and signal spatial arms. For each unresolved spectral partner the code constructs Stokes operators,

$$
\hat S_0=\hat n_H+\hat n_V,\quad
\hat S_1=\hat n_H-\hat n_V,
$$

$$
\hat S_2=\hat a_H^\dagger\hat a_V+\hat a_V^\dagger\hat a_H,
\quad
\hat S_3=-i(\hat a_H^\dagger\hat a_V-\hat a_V^\dagger\hat a_H).
$$

A linear-polarization analyzer at physical angle $\theta$ uses

$$
\hat N_\pm(\theta)
=
\frac12
\left[
\hat S_0
\pm \cos(2\theta)\hat S_1
\pm \sin(2\theta)\hat S_2
\right].
$$

For polarization-state quantities, the simulation projects onto exactly one photon in each spatial arm,

$$
\rho_{11}
=
\frac{\Pi_{11}\rho\Pi_{11}}
{\operatorname{Tr}(\Pi_{11}\rho)}.
$$

The maximal conditional two-qubit CHSH value is then computed with the Horodecki criterion,

$$
S_{\max}=2\sqrt{u_1+u_2},
$$

where $u_1,u_2$ are the two largest eigenvalues of $\Gamma^T\Gamma$.

The code intentionally limits brute-force multimode use because the post-NPBS Hilbert space grows exponentially.

In [ ]:
# Mode structure of full post-NPBS rho
@lru_cache(maxsize=None)
def a_mode(m,d,n):
    ops=[qt.qeye(d) for _ in range(n)]
    ops[m]=qt.destroy(d)
    return qt.tensor(*ops)

def hv_pairs(N_k,arm):
    N=2+4*N_k
    off=0 if arm=="idler" else N
    pairs=[(off,off+1)]
    for k in range(N_k):
        b=off+2+4*k
        pairs += [(b,b+3),(b+2,b+1)]
    return pairs

@lru_cache(maxsize=None)
def stokes_ops(d,N_k,arm):
    N=2+4*N_k
    nfull=2*N
    S0=S1=S2=S3=None
    for H,V in hv_pairs(N_k,arm):
        aH,aV=a_mode(H,d,nfull),a_mode(V,d,nfull)
        nH,nV=aH.dag()@aH,aV.dag()@aV
        s0=nH+nV
        s1=nH-nV
        s2=aH.dag()@aV+aV.dag()@aH
        s3=-1j*(aH.dag()@aV-aV.dag()@aH)
        if S0 is None:
            S0,S1,S2,S3=s0,s1,s2,s3
        else:
            S0+=s0; S1+=s1; S2+=s2; S3+=s3
    return S0,S1,S2,S3

# One-photon-per-arm postselection
@lru_cache(maxsize=None)
def one_photon_arm_projector(d,N_k,arm):
    N=2+4*N_k
    nfull=2*N
    off=0 if arm=="idler" else N
    arm_modes=set(range(off,off+N))
    P0=qt.basis(d,0)*qt.basis(d,0).dag()
    P1=qt.basis(d,1)*qt.basis(d,1).dag()
    P=None
    for occ in arm_modes:
        ops=[]
        for m in range(nfull):
            if m not in arm_modes: ops.append(qt.qeye(d))
            elif m==occ: ops.append(P1)
            else: ops.append(P0)
        term=qt.tensor(*ops)
        P=term if P is None else P+term
    return P

def postselect_one_each_arm(rho,d,N_k):
    Pi=one_photon_arm_projector(d,N_k,"idler")
    Ps=one_photon_arm_projector(d,N_k,"signal")
    P=Pi@Ps
    p=float(np.real(qt.expect(P,rho)))
    if p<1e-14: return None,0.0
    return P@rho@P/p,p

# Extract polarization moments from rho
def polarization_moments(rho,d,N_k):
    Si=stokes_ops(d,N_k,"idler")
    Ss=stokes_ops(d,N_k,"signal")
    mi=np.asarray([qt.expect(A,rho).real for A in Si],dtype=float)
    ms=np.asarray([qt.expect(A,rho).real for A in Ss],dtype=float)
    G=np.asarray([[qt.expect(A@B,rho).real for B in Ss] for A in Si],dtype=float)
    return {"mi":mi,"ms":ms,"G":G}


    
def analyzer_vector(theta,outcome="+"):
    t=np.deg2rad(theta)
    q=1 if outcome=="+" else -1
    return np.array([1.0,q*np.cos(2*t),q*np.sin(2*t),0.0])

def single_from_moments(M,theta,arm="idler",outcome="+"):
    v=analyzer_vector(theta,outcome)
    m=M["mi"] if arm=="idler" else M["ms"]
    return 0.5*v@m

def coincidence_from_moments(M,alpha,beta,pi="+",ps="+"):
    vi=analyzer_vector(alpha,pi)
    vs=analyzer_vector(beta,ps)
    return 0.25*vi@M["G"]@vs

def all_coincidences(M,alpha,beta):
    return {
        "++":coincidence_from_moments(M,alpha,beta,"+","+"),
        "+-":coincidence_from_moments(M,alpha,beta,"+","-"),
        "-+":coincidence_from_moments(M,alpha,beta,"-","+"),
        "--":coincidence_from_moments(M,alpha,beta,"-","-")
    }

# Visibility + CHSH from simulated rho
def visibility_sim(M,beta=0,n_scan=181):
    a=np.linspace(0,180,n_scan)
    C=np.array([coincidence_from_moments(M,x,beta,"+","+") for x in a])
    cmax,cmin=np.max(C),np.min(C)
    return np.nan if cmax+cmin<1e-14 else (cmax-cmin)/(cmax+cmin)

def correlator_sim(M,alpha,beta):
    C=all_coincidences(M,alpha,beta)
    den=sum(C.values())
    return np.nan if den<1e-14 else (C["++"]+C["--"]-C["+-"]-C["-+"])/den

def chsh_sim(M):
    a,ap=0,45
    b,bp=-22.5,22.5
    return abs(correlator_sim(M,a,b)+correlator_sim(M,a,bp)+correlator_sim(M,ap,b)-correlator_sim(M,ap,bp))

def chsh_max_sim(M):
    G=M["G"]
    Tcorr=G[1:4,1:4]/G[0,0]
    eig=np.linalg.eigvalsh(Tcorr.T@Tcorr)
    eig=np.sort(np.real(eig))[::-1]
    return 2*np.sqrt(max(0,eig[0]+eig[1]))

# Run YOUR state_evolution_spdc() and use returned rho
def simulate_case(d,N_k,g,T,nth,T_bs=0.5):
    precision=compute_precision(d,g,N_k)
    params=(d,N_k,g,precision,1.0,0.0,0.0,T_bs,0.0,0.0,T,nth,0.0,T,nth,0.0)
    r=state_evolution_spdc(params)
    rho=r["rho_spdc_after_bs_after_dt_full_mode_final_state"]
    expected=2*r["total_modes"]
    if len(rho.dims[0])!=expected:
        raise ValueError(f"Full rho has {len(rho.dims[0])} modes; expected {expected}. Check NPBS output ordering.")
    M_raw=polarization_moments(rho,d,N_k)
    rho_pair,p_pair=postselect_one_each_arm(rho,d,N_k)
    if rho_pair is None:
        return {"result":r,"rho":rho,"M_raw":M_raw,"M_pair":None,"p_pair":0,"V":np.nan,"S":np.nan}
    M_pair=polarization_moments(rho_pair,d,N_k)
    return {"result":r,"rho":rho,"M_raw":M_raw,"M_pair":M_pair,"p_pair":p_pair,"V":visibility_sim(M_pair),"S":chsh_max_sim(M_pair)}

# Parameters
Fk_dim=2
g = 0.04
N_measurements=1e6
beta=0
theta=np.linspace(0,180,37)

cases=[
    (0,1.00,0.00),
    (0,0.85,0.1),
    # (1,1.00,0.00),
    # (1,0.85,0.05)
]

# IMPORTANT: do not use N_k=5 with full rho.
# For Fk_dim=2, N_k=5 means 44 full modes -> dimension 2^44.
Nk_scan=[0]
nth_scan=[0.00,0.05,0.1,0.2]
T_scan=np.linspace(0.05,1,15)
cmap=plt.colormaps["tab10"]

# Simulate four requested cases
case_results={}
for Nk,T,nth in cases:
    # print(f"Running N_k={Nk}, T={T:.2f}, nth={nth:.2f}")
    case_results[(Nk,T,nth)]=simulate_case(Fk_dim,Nk,g,T,nth)

    
# Scan visibility and CHSH vs T
scan_results={}
for Nk in Nk_scan:
    for nth in nth_scan:
        V,S=[],[]
        for T in T_scan:
            # print(f"Scan: N_k={Nk}, nth={nth:.2f}, T={T:.2f}")
            r=simulate_case(Fk_dim,Nk,g,T,nth)
            V.append(r["V"])
            S.append(r["S"])
        scan_results[(Nk,nth)]={"V":np.array(V),"S":np.array(S)}

# Four simulation plots
fig,axes=plt.subplots(2,2,figsize=(15,9))
for ax,lab in zip(axes.flat,["(a)","(b)","(c)","(d)"]):
    ax.text(-0.13,1.05,lab,transform=ax.transAxes,fontsize=16,fontweight="bold",va="bottom",clip_on=False)

# (a) Singles: RAW rho
ax=axes[0,0]
for i,(Nk,T,nth) in enumerate(cases):
    r=case_results[(Nk,T,nth)]
    y=np.array([single_from_moments(r["M_raw"],a,"idler","+") for a in theta])
    label=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$"
    ax.plot(theta,N_measurements*y,'o',ms=5,color=cmap(i),label=label)
ax.set(xlabel=r"Idler analyzer angle $\alpha$ [deg.]",ylabel="Singles counts",xlim=(0,180))
ax.set_xticks(np.arange(0,181,30))

# (b) Postselected coincidence counts, but keep surviving-pair rate
ax=axes[0,1]
for i,(Nk,T,nth) in enumerate(cases):
    r=case_results[(Nk,T,nth)]
    M,p=r["M_pair"],r["p_pair"]
    HH=np.array([coincidence_from_moments(M,a,beta,"+","+") for a in theta])
    HV=np.array([coincidence_from_moments(M,a,beta,"+","-") for a in theta])
    label_HH=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{\mathrm{{th}}}}={nth:.2f},\,\mathrm{{HH}}$"
    label_HV=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{\mathrm{{th}}}}={nth:.2f},\,\mathrm{{HV}}$"
    ax.plot(theta,N_measurements*p*HH,'o',mfc='none',ms=5,color=cmap(i),label=label_HH)
    ax.plot(theta,N_measurements*p*HV,'o',ms=5,color=cmap(i),label=label_HV)
ax.set(xlabel=r"Idler analyzer angle $\alpha$ [deg.]",ylabel="Coincidence counts",xlim=(0,180))
ax.set_xticks(np.arange(0,181,30))

# (c) Visibility vs T
ax=axes[1,0]
for i,Nk in enumerate(Nk_scan):
    for nth in nth_scan:
        ax.plot(T_scan,scan_results[(Nk,nth)]["V"],'o',ms=5,label=fr"$N_k={Nk},\,\bar{{n}}_{{th}}={nth:.2f}$")
ax.axhline(1/np.sqrt(2),color='k',ls=':',lw=1.2,label=r"$V=1/\sqrt{2}$")
ax.set(xlabel=r"Transmissivity $T$",ylabel=r"Polarization visibility $V_{\rm pol}$",xlim=(0,1),ylim=(0,1.05))

# (d) CHSH vs T
ax=axes[1,1]
for i,Nk in enumerate(Nk_scan):
    for nth in nth_scan:
        ax.plot(T_scan,scan_results[(Nk,nth)]["S"],'o',ms=5,label=fr"$N_k={Nk},\,\bar{{n}}_{{th}}={nth:.2f}$")

ax.axhline(2,color='k',ls='--',lw=1.2,label=r"Local bound $S=2$")
ax.axhline(2*np.sqrt(2),color='k',ls=':',lw=1.2,label=r"Quantum bound $S=2\sqrt{2}$")
ax.set(xlabel=r"Transmissivity $T$",ylabel=r"CHSH parameter $S_{\rm CHSH}$",xlim=(0,1),ylim=(0,3))
ax.axhspan(0,2.0,facecolor="lightblue",alpha=0.3,label="CHSH-nonviolating regime")
ax.axhspan(2.0,2.8284,facecolor="lightgreen",alpha=0.25,label="CHSH-violating regime")
ax.text(0.06,1.0,"CHSH-nonviolating\nregime",fontsize=14,color="blue",alpha=0.8)
ax.text(0.06,2.25,"CHSH-violating\nregime",fontsize=14,color="green",alpha=0.8)

for ax in axes.flat:
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(which='major',direction='in',length=6,top=True,right=True,labelsize=14)
    ax.tick_params(which='minor',direction='in',length=3,top=True,right=True)
    ax.grid(which='major',ls='--',lw=0.6,alpha=0.4)
    ax.grid(which='minor',ls='--',lw=0.35,alpha=0.2)
    ax.legend(fontsize=11,frameon=True,facecolor="#F0F2F1",edgecolor="none",bbox_to_anchor=(1.05, 1))

plt.tight_layout()
plt.savefig("images/polarization_simulation_4panels.png",dpi=300,bbox_inches="tight")
plt.show()



def save_polarization_panel_csvs(case_results,scan_results,cases,Nk_scan,nth_scan,theta,T_scan,beta,N_measurements,outdir="csv_data"):
    os.makedirs(outdir,exist_ok=True)
    A=[]; B=[]; C=[]; D=[]
    for case_id,(Nk,T,nth) in enumerate(cases):
        r=case_results[(Nk,T,nth)]
        singles=np.array([single_from_moments(r["M_raw"],a,"idler","+") for a in theta])
        for a,y in zip(theta,singles):
            A.append({"case_id":case_id,"N_k":Nk,"g":g,"T":T,"n_th":nth,"alpha_deg":a,"singles_probability":y,"singles_counts":N_measurements*y})
        M,p=r["M_pair"],r["p_pair"]
        if M is None: continue
        channels=[(0,"HH","+","+"),(1,"HV","+","-")]
        for channel_id,channel,pi,ps in channels:
            values=np.array([coincidence_from_moments(M,a,beta,pi,ps) for a in theta])
            for a,y in zip(theta,values):
                probability=p*y
                B.append({"case_id":case_id,"channel_id":channel_id,"channel":channel,"N_k":Nk,"g":g,"T":T,"n_th":nth,"alpha_deg":a,"beta_deg":beta,"pair_probability":p,"coincidence_probability":probability,"coincidence_counts":N_measurements*probability})
    for scan_id,(Nk,nth) in enumerate((Nk,nth) for Nk in Nk_scan for nth in nth_scan):
        V=scan_results[(Nk,nth)]["V"]
        S=scan_results[(Nk,nth)]["S"]
        for T,v,s in zip(T_scan,V,S):
            C.append({"scan_id":scan_id,"N_k":Nk,"g":g,"n_th":nth,"T":T,"visibility":v})
            D.append({"scan_id":scan_id,"N_k":Nk,"g":g,"n_th":nth,"T":T,"S_CHSH":s})
    files={
        "polarization_simulation_singles_panel_a.csv":A,
        "polarization_simulation_coincidences_panel_b.csv":B,
        "polarization_simulation_visibility_panel_c.csv":C,
        "polarization_simulation_chsh_panel_d.csv":D}
    for name,rows in files.items():
        path=os.path.join(outdir,name)
        pd.DataFrame(rows).to_csv(path,index=False,na_rep="nan",float_format="%.12g")
        # print(f"Saved: {path}")

# Add immediately after scan_results has been calculated:
save_polarization_panel_csvs(case_results,scan_results,cases,Nk_scan,nth_scan,theta,T_scan,beta,N_measurements)

## 12. Measurement-level theory–simulation comparison

This cell places the reduced analytical model and the converged density-matrix calculation on the same four-panel figure.

The main validation point uses

$$
g=0.04,\qquad \eta=0.5,\qquad N_k=0,
$$

so the reduced weak-gain count model can be assessed independently against the finite-Fock simulation.

Important estimator distinction:

- analytical curves use $S_{\rm CHSH}^{(1)}=2\sqrt2\,V_{\rm pol}$;
- numerical markers use the Horodecki maximum $S_{\max}$ from the conditioned polarization state.

They are related diagnostics but not algebraically identical estimators in the full density-matrix model.

In [ ]:
# Theory
def pol_theory_parameters(g,T,nth,N_k,eta_bs=.5):
    g,T=np.asarray(g),np.asarray(T)
    M=1+2*N_k
    n0=np.sinh(g)**2
    b=(1-T)*nth
    Q=T**2*eta_bs*(1-eta_bs)*n0
    Bth=M**2*b*(T*n0+b)
    return M,Q,Bth

def singles_theory(g,T,nth,N_k,eta_bs=.5):
    M=1+2*N_k
    n0=np.sinh(g)**2
    return {"idler":T*eta_bs*n0+(1-T)*nth,"signal":T*(1-eta_bs)*n0+(1-T)*nth}

def coincidence_theory(alpha,beta,g,T,nth,N_k,eta_bs=.5):
    M,Q,Bth=pol_theory_parameters(g,T,nth,N_k,eta_bs)
    d=np.deg2rad(np.asarray(alpha)-beta)
    S=M*Q
    same=Bth+S*np.sin(d)**2
    cross=Bth+S*np.cos(d)**2
    return {"HH":same,"HV":cross,"VH":cross,"VV":same}

def visibility_theory(g,T,nth,N_k,eta_bs=.5):
    M,Q,Bth=pol_theory_parameters(g,T,nth,N_k,eta_bs)
    S=M*Q
    with np.errstate(divide="ignore",invalid="ignore"):
        V=S/(S+2*Bth)
    return np.where(S+2*Bth>1e-15,V,np.nan)

def chsh_theory(g,T,nth,N_k,eta_bs=.5):
    return 2*np.sqrt(2)*visibility_theory(g,T,nth,N_k,eta_bs)

# Simulation
@lru_cache(maxsize=None)
def a_mode(m,d,n):
    ops=[qt.qeye(d) for _ in range(n)]
    ops[m]=qt.destroy(d)
    return qt.tensor(*ops)

def hv_pairs(N_k,arm):
    N=2+4*N_k
    off=0 if arm=="idler" else N
    pairs=[(off,off+1)]
    for k in range(N_k):
        b=off+2+4*k
        pairs.extend([(b,b+3),(b+2,b+1)])
    return pairs

@lru_cache(maxsize=None)
def stokes_ops(d,N_k,arm):
    nfull=2*(2+4*N_k)
    S0=S1=S2=S3=None
    for H,V in hv_pairs(N_k,arm):
        aH=a_mode(H,d,nfull)
        aV=a_mode(V,d,nfull)
        nH=aH.dag()@aH
        nV=aV.dag()@aV
        s0=nH+nV
        s1=nH-nV
        s2=aH.dag()@aV+aV.dag()@aH
        s3=-1j*(aH.dag()@aV-aV.dag()@aH)
        if S0 is None:
            S0,S1,S2,S3=s0,s1,s2,s3
        else:
            S0+=s0
            S1+=s1
            S2+=s2
            S3+=s3
    return S0,S1,S2,S3

@lru_cache(maxsize=None)
def one_photon_arm_projector(d,N_k,arm):
    N=2+4*N_k
    nfull=2*N
    off=0 if arm=="idler" else N
    modes=set(range(off,off+N))
    P0=qt.basis(d,0)*qt.basis(d,0).dag()
    P1=qt.basis(d,1)*qt.basis(d,1).dag()
    P=None
    for occ in modes:
        ops=[qt.qeye(d) if m not in modes else (P1 if m==occ else P0) for m in range(nfull)]
        term=qt.tensor(*ops)
        P=term if P is None else P+term
    return P

def postselect_one_each_arm(rho,d,N_k):
    P=one_photon_arm_projector(d,N_k,"idler")@one_photon_arm_projector(d,N_k,"signal")
    p=float(np.real(qt.expect(P,rho)))
    if p<1e-14:
        return None,0.
    return P@rho@P/p,p

def polarization_moments(rho,d,N_k):
    Si=stokes_ops(d,N_k,"idler")
    Ss=stokes_ops(d,N_k,"signal")
    mi=np.asarray([qt.expect(A,rho).real for A in Si],dtype=float)
    ms=np.asarray([qt.expect(A,rho).real for A in Ss],dtype=float)
    G=np.asarray([[qt.expect(A@B,rho).real for B in Ss] for A in Si],dtype=float)
    return {"mi":mi,"ms":ms,"G":G}

def analyzer_vector(theta,outcome="+"):
    t=np.deg2rad(theta)
    q=1 if outcome=="+" else -1
    return np.array([1.,q*np.cos(2*t),q*np.sin(2*t),0.])

def single_from_moments(M,theta,arm="idler",outcome="+"):
    moments=M["mi"] if arm=="idler" else M["ms"]
    return .5*analyzer_vector(theta,outcome)@moments

def coincidence_from_moments(M,alpha,beta,pi="+",ps="+"):
    return .25*analyzer_vector(alpha,pi)@M["G"]@analyzer_vector(beta,ps)

def visibility_sim(M,beta=0,n_scan=181):
    alpha=np.linspace(0,180,n_scan)
    C=np.array([coincidence_from_moments(M,a,beta,"+","+") for a in alpha])
    cmax,cmin=C.max(),C.min()
    return np.nan if cmax+cmin<1e-14 else (cmax-cmin)/(cmax+cmin)

def chsh_max_sim(M):
    G=M["G"]
    Tcorr=G[1:4,1:4]/G[0,0]
    eig=np.sort(np.real(np.linalg.eigvalsh(Tcorr.T@Tcorr)))[::-1]
    return 2*np.sqrt(max(0,eig[0]+eig[1]))

def simulate_case(d,N_k,g,T,nth,T_bs=.5):
    p=(d,N_k,g,compute_precision(d,g,N_k),1.,0.,0.,T_bs,0.,0.,T,nth,0.,T,nth,0.)
    r=state_evolution_spdc(p)
    rho=r["rho_spdc_after_bs_after_dt_full_mode_final_state"]
    if len(rho.dims[0])!=2*r["total_modes"]:
        raise ValueError("Post-NPBS rho mode number is inconsistent.")
    M_raw=polarization_moments(rho,d,N_k)
    rho_pair,p_pair=postselect_one_each_arm(rho,d,N_k)
    if rho_pair is None:
        return {"M_raw":M_raw,"M_pair":None,"p_pair":0.,"V":np.nan,"S":np.nan}
    M_pair=polarization_moments(rho_pair,d,N_k)
    return {"M_raw":M_raw,"M_pair":M_pair,"p_pair":p_pair,"V":visibility_sim(M_pair),"S":chsh_max_sim(M_pair)}

# Parameters
Fk_dim=2
g=.04
eta_bs=.5
beta=0.
N_measurements=1e6
theta_sim=np.linspace(0,180,19)
theta_th=np.linspace(0,180,361)
cases=[(0,1.00,0.00),(0,.85,.02)]
Nk_scan=[0]
nth_scan=[0.,.02,.04,.10]
T_scan=np.linspace(0,1,15)
T_th=np.linspace(1e-4,1,500)
cmap=plt.colormaps["tab10"]
os.makedirs("csv_data",exist_ok=True)
os.makedirs("images",exist_ok=True)

def save_csv(rows,filename):
    pd.DataFrame(rows).to_csv(f"csv_data/{filename}",index=False,na_rep="nan")

# Run simulations
case_results={(Nk,T,nth):simulate_case(Fk_dim,Nk,g,T,nth) for Nk,T,nth in cases}
scan_results={}
for Nk in Nk_scan:
    for nth in nth_scan:
        results=[simulate_case(Fk_dim,Nk,g,T,nth) for T in T_scan]
        scan_results[(Nk,nth)]={"V":np.array([r["V"] for r in results]),"S":np.array([r["S"] for r in results])}

# Plot and CSV containers
fig,axes=plt.subplots(2,2,figsize=(16,8))
for ax,label in zip(axes.flat,["(a)","(b)","(c)","(d)"]):
    ax.text(-.13,1.05,label,transform=ax.transAxes,fontsize=16,fontweight="bold",va="bottom",ha="left",clip_on=False)

singles_rows=[]
coincidence_rows=[]
visibility_rows=[]
chsh_rows=[]

# (a) Singles
ax=axes[0,0]
for case_id,(Nk,T,nth) in enumerate(cases):
    r=case_results[(Nk,T,nth)]
    ysim=np.array([single_from_moments(r["M_raw"],alpha) for alpha in theta_sim])
    yth=np.full_like(theta_th,singles_theory(g,T,nth,Nk,eta_bs)["idler"],dtype=float)
    label=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$"
    ax.plot(theta_th,N_measurements*yth,"-",lw=1.4,color=cmap(case_id),label=label)
    ax.plot(theta_sim,N_measurements*ysim,"o",mfc="none",ms=5,color=cmap(case_id))
    for alpha,y in zip(theta_th,yth):
        singles_rows.append({"case_id":case_id,"source_id":0,"source":"theory","N_k":Nk,"g":g,"T":T,"n_th":nth,"alpha_deg":alpha,"singles_probability":y,"singles_counts":N_measurements*y})
    for alpha,y in zip(theta_sim,ysim):
        singles_rows.append({"case_id":case_id,"source_id":1,"source":"simulation","N_k":Nk,"g":g,"T":T,"n_th":nth,"alpha_deg":alpha,"singles_probability":y,"singles_counts":N_measurements*y})
ax.plot([],[],"ko",mfc="none",label="Simulation")
ax.set(xlabel=r"Idler analyzer angle $\alpha$ [deg.]",ylabel="Singles counts",xlim=(0,180))
ax.set_xticks(np.arange(0,181,30))
save_csv(singles_rows,"polarization_singles_panel_a.csv")

# (b) Coincidences
ax=axes[0,1]
for case_id,(Nk,T,nth) in enumerate(cases):
    r=case_results[(Nk,T,nth)]
    M=r["M_pair"]
    p_pair=r["p_pair"]
    HH_sim=np.array([coincidence_from_moments(M,alpha,beta,"+","+") for alpha in theta_sim])
    HV_sim=np.array([coincidence_from_moments(M,alpha,beta,"+","-") for alpha in theta_sim])
    Cth=coincidence_theory(theta_th,beta,g,T,nth,Nk,eta_bs)
    base=fr"$N_k={Nk},\,T={T:.2f},\,\bar{{n}}_{{th}}={nth:.2f}$"
    ax.plot(theta_th,N_measurements*Cth["HH"],"-",lw=1.4,color=cmap(case_id),label=base+r",\,\mathrm{HH}$")
    ax.plot(theta_th,N_measurements*Cth["HV"],"--",lw=1.4,color=cmap(case_id),label=base+r",\,\mathrm{HV}$")
    ax.plot(theta_sim,N_measurements*p_pair*HH_sim,"o",mfc="none",ms=5,color=cmap(case_id))
    ax.plot(theta_sim,N_measurements*p_pair*HV_sim,"s",mfc="none",ms=5,color=cmap(case_id))
    channels=[(0,"HH",Cth["HH"],HH_sim),(1,"HV",Cth["HV"],HV_sim)]
    for channel_id,channel,theory_values,sim_values in channels:
        for alpha,y in zip(theta_th,theory_values):
            coincidence_rows.append({"case_id":case_id,"source_id":0,"source":"theory","channel_id":channel_id,"channel":channel,"N_k":Nk,"g":g,"T":T,"n_th":nth,"alpha_deg":alpha,"beta_deg":beta,"pair_probability":1.,"coincidence_probability":y,"coincidence_counts":N_measurements*y})
        for alpha,y in zip(theta_sim,sim_values):
            probability=p_pair*y
            coincidence_rows.append({"case_id":case_id,"source_id":1,"source":"simulation","channel_id":channel_id,"channel":channel,"N_k":Nk,"g":g,"T":T,"n_th":nth,"alpha_deg":alpha,"beta_deg":beta,"pair_probability":p_pair,"coincidence_probability":probability,"coincidence_counts":N_measurements*probability})
ax.plot([],[],"ko",mfc="none",label=r"Simulation $HH$")
ax.plot([],[],"ks",mfc="none",label=r"Simulation $HV$")
ax.set(xlabel=r"Idler analyzer angle $\alpha$ [deg.]",ylabel="Coincidence counts",xlim=(0,180))
ax.set_xticks(np.arange(0,181,30))
save_csv(coincidence_rows,"polarization_coincidences_panel_b.csv")

# (c) Visibility versus T
ax=axes[1,0]
scan_cases=[(Nk,nth) for Nk in Nk_scan for nth in nth_scan]
for scan_id,(Nk,nth) in enumerate(scan_cases):
    Vth=visibility_theory(g,T_th,nth,Nk,eta_bs)
    Vsim=scan_results[(Nk,nth)]["V"]
    label=fr"$N_k={Nk},\,\bar{{n}}_{{th}}={nth:.2f}$"
    ax.plot(T_th,Vth,"-",lw=1.4,color=cmap(scan_id),label=label)
    ax.plot(T_scan,Vsim,"o",mfc="none",ms=5,color=cmap(scan_id))
    for T,y in zip(T_th,Vth):
        visibility_rows.append({"scan_id":scan_id,"source_id":0,"source":"theory","N_k":Nk,"g":g,"n_th":nth,"T":T,"visibility":y})
    for T,y in zip(T_scan,Vsim):
        visibility_rows.append({"scan_id":scan_id,"source_id":1,"source":"simulation","N_k":Nk,"g":g,"n_th":nth,"T":T,"visibility":y})
ax.plot([],[],"ko",mfc="none",label="Simulation")
ax.axhline(1/np.sqrt(2),color="k",ls=":",lw=1.2,label=r"$V=1/\sqrt{2}$")
ax.set(xlabel=r"Transmissivity $T$",ylabel=r"Polarization visibility $V_{\rm pol}$",xlim=(0,1),ylim=(0,1.05))
save_csv(visibility_rows,"polarization_visibility_panel_c.csv")

# (d) CHSH versus T
ax=axes[1,1]
for scan_id,(Nk,nth) in enumerate(scan_cases):
    Sth=chsh_theory(g,T_th,nth,Nk,eta_bs)
    Ssim=scan_results[(Nk,nth)]["S"]
    label=fr"$N_k={Nk},\,\bar{{n}}_{{th}}={nth:.2f}$"
    ax.plot(T_th,Sth,"-",lw=1.4,color=cmap(scan_id),label=label)
    ax.plot(T_scan,Ssim,"o",mfc="none",ms=5,color=cmap(scan_id))
    for T,y in zip(T_th,Sth):
        chsh_rows.append({"scan_id":scan_id,"source_id":0,"source":"theory","N_k":Nk,"g":g,"n_th":nth,"T":T,"S_CHSH":y})
    for T,y in zip(T_scan,Ssim):
        chsh_rows.append({"scan_id":scan_id,"source_id":1,"source":"simulation","N_k":Nk,"g":g,"n_th":nth,"T":T,"S_CHSH":y})
ax.plot([],[],"ko",mfc="none",label="Simulation")
ax.axhline(2,color="k",ls="--",lw=1.2,label=r"Local bound $S=2$")
ax.axhline(2*np.sqrt(2),color="k",ls=":",lw=1.2,label=r"Quantum bound $S=2\sqrt{2}$")
ax.axhspan(0,2,facecolor="lightblue",alpha=.3,label="CHSH-nonviolating regime")
ax.axhspan(2,2*np.sqrt(2),facecolor="lightgreen",alpha=.25,label="CHSH-violating regime")
ax.text(.06,1.,"CHSH-nonviolating\nregime",fontsize=14,color="blue",alpha=.8)
ax.text(.06,2.25,"CHSH-violating\nregime",fontsize=14,color="green",alpha=.8)
ax.set(xlabel=r"Transmissivity $T$",ylabel=r"CHSH parameter $S_{\rm CHSH}$",xlim=(0,1),ylim=(0,3))
save_csv(chsh_rows,"polarization_chsh_panel_d.csv")

# Formatting
for ax in axes.flat:
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(which="major",direction="in",length=6,top=True,right=True,labelsize=14)
    ax.tick_params(which="minor",direction="in",length=3,top=True,right=True)
    ax.grid(which="major",ls="--",lw=.6,alpha=.4)
    ax.grid(which="minor",ls="--",lw=.35,alpha=.2)
    ax.legend(fontsize=12,frameon=True,facecolor="#F5F6FF",edgecolor="none",bbox_to_anchor=(1.05,1))

plt.tight_layout()
plt.savefig("images/polarization_theory_simulation_4panels.png",dpi=300,bbox_inches="tight")
plt.show()

## 13. Cartesian total-coincidence phase maps

Summing the four polarization outcomes gives the reduced-model total coincidence moment

$$
C_{\rm tot}=4B_{\rm th}+2MQ.
$$

The correlated SPDC contribution scales linearly with $M$, while the unresolved thermal background scales as $M^2$. This cell evaluates the $(T,\bar n_{\rm th})$ phase field for representative discrete $N_k$ values and exports both fields and contour lines.

The displayed expected count is $N_{\rm meas}C_{\rm tot}$; $N_{\rm meas}$ is a deterministic plotting normalization, not stochastic detector sampling.

In [ ]:
def pol_theory_parameters(g,T,nth,Nk,eta=.5):
    M=1+2*Nk; n0=np.sinh(g)**2; b=(1-T)*nth
    return M,T**2*eta*(1-eta)*n0,M**2*b*(T*n0+b)

def coincidence_theory(alpha,beta,g,T,nth,Nk,eta=.5):
    M,Q,B=pol_theory_parameters(g,T,nth,Nk,eta); d=np.deg2rad(alpha-beta); S=M*Q
    same=B+S*np.sin(d)**2; cross=B+S*np.cos(d)**2
    return {"HH":same,"HV":cross,"VH":cross,"VV":same}

def total_counts(T,nth,Nk,g=.1,eta=.5,N_measurements=1.):
    C=coincidence_theory(0,0,g,T,nth,Nk,eta)
    return N_measurements*sum(C.values())

def save_contours(cs,Nk,path,max_points=120):
    rows=[]
    for level,segments in zip(cs.levels,cs.allsegs):
        for segment_id,segment in enumerate(segments):
            if len(segment)<2: continue
            ids=np.unique(np.linspace(0,len(segment)-1,min(len(segment),max_points),dtype=int))
            for point_id,(T,nth) in enumerate(segment[ids]):
                rows.append({"N_k":Nk,"level":level,"segment_id":segment_id,"point_id":point_id,"T":T,"n_th":nth})
            rows.append({"N_k":Nk,"level":level,"segment_id":segment_id,"point_id":-1,"T":np.nan,"n_th":np.nan})
    pd.DataFrame(rows,columns=["N_k","level","segment_id","point_id","T","n_th"]).to_csv(path,index=False,na_rep="nan",float_format="%.9g")

os.makedirs("csv_data",exist_ok=True)
os.makedirs("images",exist_ok=True)

g,eta,N_measurements=.3,.5,1e6
Nk_values=[0,5,10]
T=np.linspace(0,1,350); nth=np.linspace(0,1,350); TT,NN=np.meshgrid(T,nth)
Cmaps=[total_counts(TT,NN,Nk,g,eta,N_measurements) for Nk in Nk_values]
Cmax=max(np.nanmax(C) for C in Cmaps)
levels=np.linspace(0,Cmax,150); contour_levels=np.linspace(0,Cmax,12)[1:-1]
cmap=plt.colormaps["viridis"]; norm=colors.Normalize(0,Cmax)
n_csv=101
T_csv=np.linspace(0,1,n_csv); nth_csv=np.linspace(0,1,n_csv)
TT_csv,NN_csv=np.meshgrid(T_csv,nth_csv)
Ti=np.tile(np.arange(n_csv),n_csv); Ni=np.repeat(np.arange(n_csv),n_csv)


fig,axes=plt.subplots(1,3,figsize=(13,3.5),sharex=True,sharey=True)


for j,(ax,Nk,C) in enumerate(zip(axes,Nk_values,Cmaps)):
    C_csv=total_counts(TT_csv,NN_csv,Nk,g,eta,N_measurements)
    pd.DataFrame({"N_k":Nk,"g":g,"eta":eta,"N_measurements":N_measurements,"T_index":Ti,"n_th_index":Ni,"T":TT_csv.ravel(),"n_th":NN_csv.ravel(),
                  "C_total":C_csv.ravel()}).to_csv(f"csv_data/coincidence_phase_field_Nk{Nk}.csv",index=False,float_format="%.9g")
    cf=ax.contourf(TT,NN,C,levels=levels,cmap=cmap,norm=norm)
    cs=ax.contour(TT,NN,C,levels=contour_levels,colors="k",linewidths=.25,alpha=.3)
    save_contours(cs,Nk,f"csv_data/coincidence_phase_contours_Nk{Nk}.csv")
    ax.set_title(fr"$N_k={Nk}$",fontsize=16); ax.set(xlim=(0,1),ylim=(0,1),xlabel=r"Transmissivity $T$")
    ax.xaxis.label.set_size(15); ax.text(-.10,1.15,f"({chr(97+j)})",transform=ax.transAxes,fontsize=15,fontweight="bold")
    ax.xaxis.set_minor_locator(AutoMinorLocator(2)); ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(which="major",direction="in",length=5,top=True,right=True,labelsize=11)
    ax.tick_params(which="minor",direction="in",length=3,top=True,right=True)
axes[0].set_ylabel(r"Mean thermal photon number $\bar n_{\rm th}$",fontsize=14)
cbar=fig.colorbar(cf,ax=axes.ravel().tolist(),fraction=.022,pad=.025)
cbar.set_label("Expected total coincidence counts",fontsize=14); cbar.ax.tick_params(labelsize=11)
plt.savefig("images/coincidence_counts_T_nth_Nk.png",dpi=400,bbox_inches="tight")
plt.show()

## 14. Ternary total-coincidence map

This analytical ternary map visualizes the same total-coincidence model on a constrained simplex. It is useful for showing the competition between:

- pair-dominated coincidences;
- thermal accidental coincidences;
- the $M^2$ multimode enhancement of the unresolved background.

The triangular field and contour paths are exported as CSV so the same surface can be reproduced in PGFPlots/LaTeX.

In [ ]:
def ternary_pin(ax,p,pt,text,fontsize=11):
    t,l,r=p; tt,ll,rr=pt
    ax.plot([t,tt],[l,ll],[r,rr],color="white",lw=.8,zorder=8)
    ax.plot(t,l,r,"o",ms=4,mfc="white",mec="k",mew=.9,zorder=10)
    ax.text(tt,ll,rr,text,fontsize=fontsize,color="white",ha="left",va="center",zorder=10)

def physical_to_ternary(Nk,T,nth):
    a,b,c=Nk/10,T,nth; s=a+b+c
    return a/s,b/s,c/s

def ternary_grid(n):
    a,b,c=[],[],[]
    for i in range(n):
        x=i/(n-1)
        for j in range(n-i):
            y=j/(n-1); a.append(x); b.append(y); c.append(1-x-y)
    return np.array(a),np.array(b),np.array(c)

def triangles(n):
    idx=lambda i,j:i*n-i*(i-1)//2+j
    out=[]
    for i in range(n-1):
        for j in range(n-i-1):
            out.append([idx(i,j),idx(i,j+1),idx(i+1,j)])
            if j<n-i-2: out.append([idx(i,j+1),idx(i+1,j+1),idx(i+1,j)])
    return np.asarray(out)

def coincidence_total(g,T,nth,Nk,eta=.5,Nmeas=1000):
    M=1+2*Nk; n0=np.sinh(g)**2; b=(1-T)*nth
    Q=T**2*eta*(1-eta)*n0; B=M**2*b*(T*n0+b)
    return Nmeas*(4*B+2*M*Q)

def save_contours(cs,path,max_points=150):
    rows=[]
    for level,segments in zip(cs.levels,cs.allsegs):
        for segment_id,segment in enumerate(segments):
            if len(segment)<2: continue
            ids=np.unique(np.linspace(0,len(segment)-1,min(len(segment),max_points),dtype=int))
            for point_id,(x,y) in enumerate(segment[ids]):
                rows.append({"level":level,"segment":segment_id,"point":point_id,"x":x,"y":y})
            rows.append({"level":np.nan,"segment":np.nan,"point":np.nan,"x":np.nan,"y":np.nan})
    pd.DataFrame(rows,columns=["level","segment","point","x","y"]).to_csv(path,index=False,na_rep="nan",float_format="%.9g")

os.makedirs("csv_data",exist_ok=True)
g,Nmeas,eta=.3,1e1,.5
h=np.sqrt(3)/2

# High-resolution grid: fixes the common color scale and contour paths.
n=151
xNk_hi,xT_hi,xn_hi=ternary_grid(n); Nk_hi,T_hi,nth_hi=10*xNk_hi,xT_hi,xn_hi
x_hi=.5*xNk_hi+xn_hi; y_hi=h*xNk_hi
C_hi=np.nan_to_num(coincidence_total(g,T_hi,nth_hi,Nk_hi,eta,Nmeas),nan=0)
Cmax=C_hi.max(); clevels=np.linspace(0,Cmax,12)[1:-1]
fig,ax=plt.subplots()
cs=ax.tricontour(mtri.Triangulation(x_hi,y_hi,triangles(n)),C_hi,levels=clevels)
save_contours(cs,"csv_data/ternary_coincidence_contours.csv")
plt.close(fig)

# Reduced patch grid for fast Overleaf compilation.
n_csv=61
xNk,xT,xn=ternary_grid(n_csv); Nk,T,nth=10*xNk,xT,xn
x=.5*xNk+xn; y=h*xNk
C=np.nan_to_num(coincidence_total(g,T,nth,Nk,eta,Nmeas),nan=0)
rows=[]
for triangle_id,triangle in enumerate(triangles(n_csv)):
    for vertex,k in enumerate(triangle):
        rows.append({"triangle":triangle_id,"vertex":vertex,"x":x[k],"y":y[k],"N_k":Nk[k],"T":T[k],"n_th":nth[k],"g":g,"N_measurements":Nmeas,"C_total":C[k]})
pd.DataFrame(rows).to_csv("csv_data/ternary_coincidence_field.csv",index=False,float_format="%.9g")
print(f"Saved field and contours; Cmax={Cmax:.9g}")

# Recreate, save, and display the Matplotlib/mpltern figure.
try:
    import mpltern  # registers projection="ternary"
except ImportError:
    print("Plot skipped: install mpltern with `pip install mpltern`.")
else:
    os.makedirs("images",exist_ok=True)
    fig=plt.figure(figsize=(6.6,5.7))
    ax=fig.add_subplot(111,projection="ternary")
    cf=ax.tricontourf(xNk_hi,xT_hi,xn_hi,C_hi,levels=np.linspace(0,Cmax,180),cmap="magma",norm=colors.Normalize(0,Cmax))
    ax.tricontour(xNk_hi,xT_hi,xn_hi,C_hi,levels=clevels,colors="k",linewidths=.25,alpha=.30)
    ternary_pin(ax,physical_to_ternary(.2,.95,.02),(.06,.84,.10),"Pair-dominated\ncoincidences")
    ternary_pin(ax,physical_to_ternary(3,.30,.70),(.15,.40,.65),"Thermal-accidental\ndominated")
    ternary_pin(ax,physical_to_ternary(8,.45,.35),(.90,.47,.11),"Multimode-enhanced\nbackground")
    ticks=np.linspace(0,1,6)
    ax.set_tlabel(r"Effective multimode parameter $N_k$",fontsize=13)
    ax.taxis.set_ticks(ticks,labels=[f"{10*v:.0f}" for v in ticks])
    ax.set_llabel(r"Transmissivity $T$",fontsize=13)
    ax.laxis.set_ticks(ticks,labels=[f"{v:.1f}" for v in ticks])
    ax.set_rlabel(r"Mean thermal photon number $\bar n_{\rm th}$",fontsize=13)
    ax.raxis.set_ticks(ticks,labels=[f"{v:.1f}" for v in ticks])
    for axis in (ax.taxis,ax.laxis,ax.raxis):
        axis.set_label_position("tick1"); axis.set_ticks_position("tick1")
    ax.tick_params(labelsize=12,direction="in")
    cbar=fig.colorbar(cf,ax=ax,fraction=.03,pad=.10,ticks=np.arange(0,Cmax,200))
    cbar.set_label("Expected total coincidence counts",fontsize=13)
    cbar.ax.tick_params(labelsize=11)
    plt.tight_layout()
    plt.savefig("images/ternary_coincidence_counts.png",dpi=400,bbox_inches="tight")
    plt.show()



    

## 15. Conditional-CHSH phase diagram

The reduced-model CHSH boundary is obtained from

$$
S_{\rm CHSH}^{(1)}=2
\quad\Longleftrightarrow\quad
B_{\rm th}
=
\frac{\sqrt2-1}{2}MQ.
$$

Equivalently,

$$
M(1-T)\bar n_{\rm th}
\left[
T\sinh^2g+(1-T)\bar n_{\rm th}
\right]
=
\frac{\sqrt2-1}{2}
T^2\eta(1-\eta)\sinh^2g.
$$

This cell plots the $(T,\bar n_{\rm th})$ boundary for several **integer** $N_k$ values. The zero-coincidence point, where the normalized CHSH quantity is undefined, is excluded from physical interpretation.

In [ ]:
def phase_pin(ax,T,nth,Tt,ntht,text,fontsize=12):
    ax.plot(T,nth,'o',ms=4,mfc='white',mec='k',mew=.9,zorder=10)
    ax.annotate(text,xy=(T,nth),xytext=(Tt,ntht),fontsize=fontsize,color='white',ha='left',va='center',arrowprops=dict(arrowstyle='-',lw=.8,color='white'),zorder=10)

def save_contours(contour_sets,Nk,filename):
    rows=[]
    for kind_id,kind,cs in contour_sets:
        for level,segments in zip(cs.levels,cs.allsegs):
            for segment_id,segment in enumerate(segments):
                for point_id,(T,nth) in enumerate(segment):
                    rows.append({"N_k":Nk,"kind_id":kind_id,"kind":kind,"level":level,"segment_id":segment_id,"point_id":point_id,"T":T,"n_th":nth})
                rows.append({"N_k":Nk,"kind_id":kind_id,"kind":kind,"level":level,"segment_id":segment_id,"point_id":-1,"T":np.nan,"n_th":np.nan})
    pd.DataFrame(rows).to_csv(filename,index=False,na_rep="nan")

os.makedirs("csv_data",exist_ok=True)
g,eta_bs=0.3,.5
Smax=2*np.sqrt(2)
T=np.linspace(.001,1,350)
nth=np.linspace(0,1,350)
TT,NN=np.meshgrid(T,nth)
Nk_values=[0,1,7]
levels=np.linspace(0,Smax,120)
contours=np.arange(.25,2.76,.25)
cmap=plt.colormaps["jet"]
norm=colors.Normalize(0,Smax)
n_csv= 101
T_csv=np.linspace(.001,1,n_csv)
nth_csv=np.linspace(0,1,n_csv)
TT_csv,NN_csv=np.meshgrid(T_csv,nth_csv)
Ti=np.tile(np.arange(n_csv),n_csv)
Ni=np.repeat(np.arange(n_csv),n_csv)



fig,axes=plt.subplots(1,3,figsize=(13,3.5),sharex=True,sharey=True)
for j,(ax,Nk) in enumerate(zip(axes,Nk_values)):
    # High-resolution data for Matplotlib plot and contours
    S=np.clip(np.nan_to_num(
        chsh_theory(g,TT,NN,Nk,eta_bs),nan=0),0,Smax)

    # Reduced-resolution data for Overleaf CSV
    S_csv=np.clip(np.nan_to_num(
        chsh_theory(g,TT_csv,NN_csv,Nk,eta_bs),nan=0),0,Smax)

    pd.DataFrame({
        "N_k":Nk,
        "g":g,
        "eta_bs":eta_bs,
        "T_index":Ti,
        "n_th_index":Ni,
        "T":TT_csv.ravel(),
        "n_th":NN_csv.ravel(),
        "S_CHSH":S_csv.ravel()
    }).to_csv(
        f"csv_data/chsh_phase_field_Nk{Nk}.csv",
        index=False
    )

    cf=ax.contourf(
        TT,NN,S,
        levels=levels,
        cmap=cmap,
        norm=norm
    )

    cmain=ax.contour(
        TT,NN,S,
        levels=contours,
        colors='k',
        linewidths=.25,
        alpha=.35
    )

    cbound=ax.contour(
        TT,NN,S,
        levels=[2],
        colors='white',
        linewidths=1.8,
        linestyles='--'
    )

    save_contours(
        [(0,"regular",cmain),(1,"CHSH_boundary",cbound)],
        Nk,
        f"csv_data/chsh_phase_contours_Nk{Nk}.csv"
    )
    
    

    ax.set_title(fr"$N_k={Nk}$",fontsize=15)
    ax.set(xlim=(0,1),ylim=(0,1),xlabel=r"Transmissivity $T$")
    ax.xaxis.label.set_size(14)
    ax.text(-.10,1.2,f"({chr(97+j)})",transform=ax.transAxes,fontsize=15,fontweight='bold')
    ax.xaxis.set_minor_locator(AutoMinorLocator(2))
    ax.yaxis.set_minor_locator(AutoMinorLocator(2))
    ax.tick_params(which='major',direction='in',length=5,top=True,right=True,labelsize=11)
    ax.tick_params(which='minor',direction='in',length=3,top=True,right=True)

axes[0].set_ylabel(r"Mean thermal photon number $\bar n_{\rm th}$",fontsize=14)
phase_pin(axes[0],.97,.05,.46,.20,"Strong CHSH\nviolation")
phase_pin(axes[0],.91,.38,.48,.52,r"$S_{\rm CHSH}=2$"+"\nboundary")
phase_pin(axes[1],.32,.75,.42,.84,"Noise-dominated")
phase_pin(axes[2],.38,.72,.48,.82,"Multimode/noise\nlimited")

cbar=fig.colorbar(cf,ax=axes.ravel().tolist(),fraction=.022,pad=.025)
cbar.set_label(r"CHSH parameter $S_{\rm CHSH}$",fontsize=14)
cbar.set_ticks([0,.5,1,1.5,2,2.5,Smax])
cbar.set_ticklabels(["0","0.5","1.0","1.5","2.0","2.5",r"$2\sqrt{2}$"])
cbar.ax.tick_params(labelsize=11)
plt.savefig("images/CHSH_phase_diagram_slices.png",dpi=400,bbox_inches="tight")
plt.show()

## 16. Ternary conditional-CHSH map

The CHSH phase field is evaluated on the constrained ternary slice with continuous $N_k$ interpolation used only for visualization.

The white dashed contour marks $S_{\rm CHSH}^{(1)}=2$. Increasing thermal occupation or the interpolated multimode coordinate increases the relative background contribution and pushes the system toward the CHSH-nonviolating region.

In [ ]:
def ternary_pin(ax,p,pt,text,fontsize=11):
    t,l,r=p; tt,ll,rr=pt
    ax.plot([t,tt],[l,ll],[r,rr],color="white",lw=.8,zorder=8)
    ax.plot(t,l,r,"o",ms=4,mfc="white",mec="k",mew=.9,zorder=10)
    ax.text(tt,ll,rr,text,fontsize=fontsize,color="white",ha="left",va="center",zorder=10)

def physical_to_ternary(Nk,T,nth):
    a,b,c=Nk/10,T,nth/2; s=a+b+c
    return a/s,b/s,c/s

def ternary_grid(n):
    a,b,c=[],[],[]
    for i in range(n):
        x=i/(n-1)
        for j in range(n-i):
            y=j/(n-1); a.append(x); b.append(y); c.append(1-x-y)
    return np.array(a),np.array(b),np.array(c)

def triangles(n):
    idx=lambda i,j:i*n-i*(i-1)//2+j
    out=[]
    for i in range(n-1):
        for j in range(n-i-1):
            out.append([idx(i,j),idx(i,j+1),idx(i+1,j)])
            if j<n-i-2: out.append([idx(i,j+1),idx(i+1,j+1),idx(i+1,j)])
    return np.asarray(out)

def pol_theory_parameters(g,T,nth,Nk,eta=.5):
    M=1+2*Nk; n0=np.sinh(g)**2; b=(1-T)*nth
    return M,T**2*eta*(1-eta)*n0,M**2*b*(T*n0+b)

def visibility_theory(g,T,nth,Nk,eta=.5):
    M,Q,B=pol_theory_parameters(g,T,nth,Nk,eta); A=M*Q
    with np.errstate(divide="ignore",invalid="ignore"): V=A/(A+2*B)
    return np.where(A+2*B>1e-15,V,np.nan)

def chsh_theory(g,T,nth,Nk,eta=.5):
    return 2*np.sqrt(2)*visibility_theory(g,T,nth,Nk,eta)

def save_contours(sets,path,max_points=160):
    rows=[]
    for kind_id,kind,cs in sets:
        for level,segments in zip(cs.levels,cs.allsegs):
            for segment_id,segment in enumerate(segments):
                if len(segment)<2: continue
                ids=np.unique(np.linspace(0,len(segment)-1,min(len(segment),max_points),dtype=int))
                for point_id,(x,y) in enumerate(segment[ids]):
                    rows.append({"kind_id":kind_id,"kind":kind,"level":level,"segment":segment_id,"point":point_id,"x":x,"y":y})
                rows.append({"kind_id":np.nan,"kind":"","level":np.nan,"segment":np.nan,"point":np.nan,"x":np.nan,"y":np.nan})
    pd.DataFrame(rows).to_csv(path,index=False,na_rep="nan",float_format="%.9g")

os.makedirs("csv_data",exist_ok=True)
g,eta,Smax=0.3,.5,2*np.sqrt(2); h=np.sqrt(3)/2

# High-resolution field and contour paths.
n=151
xNk_hi,xT_hi,xn_hi=ternary_grid(n); Nk_hi,T_hi,nth_hi=10*xNk_hi,xT_hi,2*xn_hi
x_hi=.5*xNk_hi+xn_hi; y_hi=h*xNk_hi
S_hi=np.clip(np.nan_to_num(chsh_theory(g,T_hi,nth_hi,Nk_hi,eta),nan=0),0,Smax)
tri=mtri.Triangulation(x_hi,y_hi,triangles(n))
fig0,ax0=plt.subplots()
regular=ax0.tricontour(tri,S_hi,levels=np.arange(.25,2.76,.25))
boundary=ax0.tricontour(tri,S_hi,levels=[2])
save_contours([(0,"regular",regular),(1,"CHSH_boundary",boundary)],"csv_data/ternary_chsh_phase_contours.csv")
plt.close(fig0)

# Reduced triangular patches for fast PGFPlots compilation.
n_csv=61
xNk,xT,xn=ternary_grid(n_csv); Nk,T,nth=10*xNk,xT,2*xn
x=.5*xNk+xn; y=h*xNk
S=np.clip(np.nan_to_num(chsh_theory(g,T,nth,Nk,eta),nan=0),0,Smax)
rows=[]
for triangle_id,triangle in enumerate(triangles(n_csv)):
    for vertex,k in enumerate(triangle):
        rows.append({"triangle":triangle_id,"vertex":vertex,"x":x[k],"y":y[k],"N_k":Nk[k],"T":T[k],"n_th":nth[k],"g":g,"S_CHSH":S[k]})
pd.DataFrame(rows).to_csv("csv_data/ternary_chsh_phase_field.csv",index=False,float_format="%.9g")

# Save and display the original mpltern figure.
try:
    import mpltern
except ImportError:
    print("CSV files saved. Install mpltern to display the plot: pip install mpltern")
else:
    os.makedirs("images",exist_ok=True)
    fig=plt.figure(figsize=(6.6,5.7)); ax=fig.add_subplot(111,projection="ternary")
    cf=ax.tricontourf(xNk_hi,xT_hi,xn_hi,S_hi,levels=np.linspace(0,Smax,180),cmap="viridis",norm=colors.Normalize(0,Smax))
    ax.tricontour(xNk_hi,xT_hi,xn_hi,S_hi,levels=np.arange(.25,2.76,.25),colors="k",linewidths=.25,alpha=.3)
    ax.tricontour(xNk_hi,xT_hi,xn_hi,S_hi,levels=[2],colors="white",linewidths=2,linestyles="--")
    ternary_pin(ax,physical_to_ternary(.2,.95,.02),(.06,.84,.10),"Strong CHSH\nviolation")
    ternary_pin(ax,physical_to_ternary(1,.82,.08),(.20,.62,.18),r"$S_{\rm CHSH}=2$"+"\nboundary")
    ternary_pin(ax,physical_to_ternary(3,.30,1.2),(.22,.40,.90),"Noise-dominated\nregime")
    ternary_pin(ax,physical_to_ternary(8,.55,.20),(.68,.24,.08),"Multimode\npenalty")
    ticks=np.linspace(0,1,6)
    ax.set_tlabel(r"Effective multimode parameter $N_k$",fontsize=13); ax.taxis.set_ticks(ticks,labels=[f"{10*v:.0f}" for v in ticks])
    ax.set_llabel(r"Transmissivity $T$",fontsize=13); ax.laxis.set_ticks(ticks,labels=[f"{v:.1f}" for v in ticks])
    ax.set_rlabel(r"Mean thermal photon number $\bar n_{\rm th}$",fontsize=13); ax.raxis.set_ticks(ticks,labels=[f"{2*v:.1f}" for v in ticks])
    for axis in (ax.taxis,ax.laxis,ax.raxis): axis.set_label_position("tick1"); axis.set_ticks_position("tick1")
    ax.tick_params(labelsize=13,direction="in")
    cbar=fig.colorbar(cf,ax=ax,fraction=.03,pad=.10); cbar.set_label(r"CHSH parameter $S_{\rm CHSH}$",fontsize=13)
    cbar.set_ticks([0,.5,1,1.5,2,2.5,Smax]); cbar.set_ticklabels(["0","0.5","1.0","1.5","2.0","2.5",r"$2\sqrt{2}$"])
    plt.tight_layout(); plt.savefig("images/ternary_CHSH_phase_diagram.png",dpi=400,bbox_inches="tight"); plt.show()

print(f"Saved field and contour CSV files; Smax={Smax:.9g}")

## 17. State-level finite-Fock convergence

The analytical Gaussian result is independent of the local cutoff $d$, whereas the density-matrix model uses

$$
\mathcal H_d
=
\operatorname{span}
\{|0\rangle,\ldots,|d-1\rangle\}.
$$

For a single TMSV pair,

$$
\epsilon_{\rm SPDC}(d)=\tanh^{2d}g.
$$

For a single thermal environment,

$$
\epsilon_{\rm th}(d)
=
\left(
\frac{\bar n_{\rm th}}{1+\bar n_{\rm th}}
\right)^d.
$$

The definitive convergence test compares both the theory–simulation discrepancy

$$
\Delta_O^{(d)}
=
|O^{(d)}-O_{\rm th}|
$$

and the successive-cutoff change

$$
\delta_O^{(d_j)}
=
|O^{(d_j)}-O^{(d_{j-1})}|.
$$

**Source reconciliation note.** The uploaded exploratory notebook had `d_values = [2, 3, 4]` in this cell, while the manuscript and Supplemental Material report the publication convergence study over $d=2,\ldots,6$. The cleaned notebook below uses the manuscript range so that the reported convergence table can be regenerated. The $d=5,6$ runs are computationally expensive.

In [ ]:
def purity_fidelity_theory(g, T, nth, N_k):
    g = np.asarray(g)
    T = np.asarray(T)
    M = 1 + 2 * N_k
    c = np.cosh(2 * g)
    s = np.sinh(2 * g)
    A = T * c + (1 - T) * (2 * nth + 1)
    C = T * s
    purity_pair = 1 / (A ** 2 - C ** 2)
    fidelity_pair = 4 / ((c + A) ** 2 - (s + C) ** 2)
    return (purity_pair ** M, fidelity_pair ** M)

def run_fig1_for_dimension(Fk_dim, N_k=0, g_values=np.linspace(0, 1, 15), T_nth_pairs=((1.0, 0.08), (0.9, 0.08), (0.8, 0.08)), g_fixed=0.3, T_values=np.linspace(0, 1, 15), nth_values=(0.0, 0.02, 0.08)):
    T_bs = 0.5
    nth_bs = 0.0
    phi_bs = 0.0
    T_dt_for_bs_values_full_mode = [1.0]
    nth_dt_for_bs_values_full_mode = [0.0]
    phi_dt_for_bs_full_mode = 0.0
    T_dt_for_bs_values = [1.0]
    nth_dt_for_bs_values = [0.0]
    phi_dt_for_bs = 0.0
    phi_dt_for_spdc = 0.0
    n_jobs = max(1, mp.cpu_count() - 1)
    params_g = []
    for g in g_values:
        precision = compute_precision(Fk_dim, g, N_k)
        for T_spdc, nth_spdc in T_nth_pairs:
            for T_full in T_dt_for_bs_values_full_mode:
                for nth_full in nth_dt_for_bs_values_full_mode:
                    for T_out in T_dt_for_bs_values:
                        for nth_out in nth_dt_for_bs_values:
                            params_g.append((Fk_dim, N_k, g, precision, T_spdc, nth_spdc, phi_dt_for_spdc, T_bs, phi_bs, nth_bs, T_full, nth_full, phi_dt_for_bs_full_mode, T_out, nth_out, phi_dt_for_bs))
    results_g = Parallel(n_jobs=n_jobs, backend='loky')((delayed(state_evolution_spdc)(p) for p in params_g))
    params_T = []
    precision_T = compute_precision(Fk_dim, g_fixed, N_k)
    for T_spdc in T_values:
        for nth_spdc in nth_values:
            for T_full in T_dt_for_bs_values_full_mode:
                for nth_full in nth_dt_for_bs_values_full_mode:
                    for T_out in T_dt_for_bs_values:
                        for nth_out in nth_dt_for_bs_values:
                            params_T.append((Fk_dim, N_k, g_fixed, precision_T, T_spdc, nth_spdc, phi_dt_for_spdc, T_bs, phi_bs, nth_bs, T_full, nth_full, phi_dt_for_bs_full_mode, T_out, nth_out, phi_dt_for_bs))
    results_T = Parallel(n_jobs=n_jobs, backend='loky')((delayed(state_evolution_spdc)(p) for p in params_T))
    return {'g': results_g, 'T': results_T}

def extract_state_points(result_set, N_k):
    mode = 2 + 4 * N_k - 1
    points = {}
    for scan_name in ['g', 'T']:
        for r in result_set[scan_name]:
            if r is None:
                continue
            if r['N_k'] != N_k:
                continue
            g = float(r['g'])
            T = float(r['T_dt_for_spdc'])
            nth = float(r['nth_dt_for_spdc'])
            P_sim = float(np.real(r['purities_spdc_after_dt'][mode]))
            F_sim = float(np.real(r['fidelities_spdc_after_dt'][mode]))
            P_th, F_th = purity_fidelity_theory(g, T, nth, N_k)
            P_th = float(np.asarray(P_th))
            F_th = float(np.asarray(F_th))
            key = (scan_name, round(g, 12), round(T, 12), round(nth, 12))
            points[key] = {'P': P_sim, 'F': F_sim, 'Pth': P_th, 'Fth': F_th, 'g': g, 'T': T, 'nth': nth, 'scan': scan_name}
    return points

def make_convergence_table(results_by_d, N_k=0, g_max=1.0, nth_max=0.08):
    maps = {d: extract_state_points(results_by_d[d], N_k) for d in sorted(results_by_d)}
    rows = []
    d_sorted = sorted(maps)
    for j, d in enumerate(d_sorted):
        current = maps[d]
        P_err = np.array([abs(v['P'] - v['Pth']) for v in current.values()])
        F_err = np.array([abs(v['F'] - v['Fth']) for v in current.values()])
        if j == 0:
            step_P = np.nan
            step_F = np.nan
        else:
            d_previous = d_sorted[j - 1]
            previous = maps[d_previous]
            common = sorted(set(current.keys()) & set(previous.keys()))
            step_P = max((abs(current[k]['P'] - previous[k]['P']) for k in common))
            step_F = max((abs(current[k]['F'] - previous[k]['F']) for k in common))
        M = 1 + 2 * N_k
        Nmode = 2 + 4 * N_k
        eps_pair = np.tanh(g_max) ** (2 * d)
        eps_spdc_total = 1 - (1 - eps_pair) ** M
        qth = nth_max / (1 + nth_max)
        eps_env_total = 1 - (1 - qth ** d) ** Nmode
        worst_P_key = max(current, key=lambda k: abs(current[k]['P'] - current[k]['Pth']))
        worst_F_key = max(current, key=lambda k: abs(current[k]['F'] - current[k]['Fth']))
        wp = current[worst_P_key]
        wf = current[worst_F_key]
        rows.append({'d': d, 'max_error_P': np.max(P_err), 'RMSE_P': np.sqrt(np.mean(P_err ** 2)), 'max_error_F': np.max(F_err), 'RMSE_F': np.sqrt(np.mean(F_err ** 2)), 'max_step_P': step_P, 'max_step_F': step_F, 'eps_SPDC': eps_spdc_total, 'eps_env': eps_env_total, 'worst_P_g': wp['g'], 'worst_P_T': wp['T'], 'worst_P_nth': wp['nth'], 'worst_F_g': wf['g'], 'worst_F_T': wf['T'], 'worst_F_nth': wf['nth']})
    return pd.DataFrame(rows)

def plot_state_convergence(df):
    d = df['d'].values
    fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.4))
    axes[0].plot(d, np.maximum(df['max_error_P'].values, 1e-16), 'o-', markersize=6, linewidth=1.5, label='Purity')
    axes[0].plot(d, np.maximum(df['max_error_F'].values, 1e-16), 's-', markersize=6, linewidth=1.5, label='Fidelity')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Local Fock dimension $d$', fontsize=13)
    axes[0].set_ylabel('$\\max|\\mathcal{O}^{(d)}-\\mathcal{O}_{\\rm th}|$', fontsize=13)
    axes[0].set_title('Theory-simulation error', fontsize=13, pad=8)
    axes[0].set_xticks(d)
    axes[0].xaxis.set_minor_locator(AutoMinorLocator(4))
    valid = np.isfinite(df['max_step_P'].values)
    d_valid = d[valid]
    axes[1].plot(d_valid, np.maximum(df.loc[valid, 'max_step_P'].values, 1e-16), 'o-', markersize=6, linewidth=1.5, label='Purity')
    axes[1].plot(d_valid, np.maximum(df.loc[valid, 'max_step_F'].values, 1e-16), 's-', markersize=6, linewidth=1.5, label='Fidelity')
    axes[1].set_yscale('log')
    axes[1].set_xlabel('Successive local Fock cutoff', fontsize=13)
    axes[1].set_ylabel('$\\max|\\mathcal{O}^{(d)}-\\mathcal{O}^{(d_{\\rm prev})}|$', fontsize=13)
    axes[1].set_title('Successive-cutoff change', fontsize=13, pad=8)
    axes[1].set_xticks(d_valid)
    transition_labels = []
    d_all = df['d'].values
    for upper_d in d_valid:
        idx = np.where(d_all == upper_d)[0][0]
        lower_d = d_all[idx - 1]
        transition_labels.append(f'${lower_d}\\!\\to\\!{upper_d}$')
    axes[1].set_xticklabels(transition_labels, fontsize=11)
    axes[1].xaxis.set_minor_locator(AutoMinorLocator(4))
    for ax, panel_label in zip(axes, ['(a)', '(b)']):
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10) * 0.1, numticks=100))
        ax.yaxis.set_minor_formatter(NullFormatter())
        ax.text(-0.16, 1.06, panel_label, transform=ax.transAxes, fontsize=14, fontweight='bold', ha='left', va='bottom', clip_on=False)
        ax.tick_params(axis='both', which='major', direction='in', length=6, width=1.0, top=True, right=True, labelsize=11)
        ax.tick_params(axis='both', which='minor', direction='in', length=3, width=0.8, top=True, right=True)
        ax.grid(True, which='major', linestyle='--', linewidth=0.7, alpha=0.45)
        ax.grid(True, which='minor', linestyle='--', linewidth=0.4, alpha=0.2)
        ax.legend(frameon=False, fontsize=11, loc='best')
    plt.tight_layout()
    plt.savefig('images/state_fock_convergence.png', dpi=400, bbox_inches='tight')
    plt.show()





N_k = 0
d_values = [2, 3, 4, 5, 6]
results_by_d = {}
for d in d_values:
    print(f'\nRunning d = {d}\n')
    results_by_d[d] = run_fig1_for_dimension(Fk_dim=d, N_k=N_k)
df_conv = make_convergence_table(results_by_d, N_k=N_k, g_max=1.0, nth_max=0.08)
print('\nFull convergence table:\n')
print(df_conv.to_string(index=False))
df_conv.to_csv('images/state_fock_convergence_table.csv', index=False)
latex_columns = ['d', 'max_error_P', 'RMSE_P', 'max_error_F', 'RMSE_F', 'max_step_P', 'max_step_F', 'eps_SPDC', 'eps_env']
print('\nLaTeX convergence table:\n')
print(df_conv[latex_columns].to_latex(index=False, float_format=lambda x: f'{x:.3e}'))


plot_state_convergence(df_conv)

## 18. Measurement-level finite-Fock convergence

The polarization benchmark uses a normalized convergence metric because singles, coincidence moments, visibility, and CHSH have different natural scales.

For observable $O$,

$$
\widehat{\Delta}^{(d)}_O
=
\frac{
\max_\lambda
|O_{\rm sim}^{(d)}(\lambda)-O_{\rm th}(\lambda)|
}{
O_{\rm scale}
},
$$

$$
\widehat{\delta}^{(d_{\rm prev}\to d)}_O
=
\frac{
\max_\lambda
|O_{\rm sim}^{(d)}(\lambda)-O_{\rm sim}^{(d_{\rm prev})}(\lambda)|
}{
O_{\rm scale}
}.
$$

The manuscript reports that the $4\to5$ refinement is below $10^{-3}$ for singles, coincidences, $V_{\rm pol}$, and $S_{\max}$; $d=4$ is therefore sufficient for the weak-gain measurement benchmark, and $d=5,6$ provide stability checks.

As above, the uploaded exploratory notebook listed only $d=2,3,4$, while the manuscript reports $d=2,\ldots,6$. This cleaned version uses the full publication range.

In [ ]:
# THEORY
def pol_theory_parameters(g, T, nth, N_k, eta_bs=0.5):
    g = np.asarray(g)
    T = np.asarray(T)
    M = 1 + 2*N_k
    n0 = np.sinh(g)**2
    b = (1-T)*nth
    Q = T**2*eta_bs*(1-eta_bs)*n0
    Bth = M**2*b*(T*n0+b)
    return M, Q, Bth

def singles_theory(g, T, nth, N_k, eta_bs=0.5):
    M = 1 + 2*N_k
    n0 = np.sinh(g)**2
    return {"idler": M*(T*eta_bs*n0 + (1-T)*nth), "signal": M*(T*(1-eta_bs)*n0 + (1-T)*nth)}

def coincidence_theory(alpha, beta, g, T, nth, N_k, eta_bs=0.5):
    M, Q, Bth = pol_theory_parameters(g, T, nth, N_k, eta_bs)
    dtheta = np.deg2rad(np.asarray(alpha)-beta)
    S = M*Q
    same = Bth + S*np.sin(dtheta)**2
    cross = Bth + S*np.cos(dtheta)**2
    return {"HH": same, "HV": cross, "VH": cross, "VV": same}

def visibility_theory(g, T, nth, N_k, eta_bs=0.5):
    M, Q, Bth = pol_theory_parameters(g, T, nth, N_k, eta_bs)
    S = M*Q
    denominator = S + 2*Bth
    with np.errstate(divide="ignore", invalid="ignore"):
        V = S/denominator
    return np.where(denominator > 1e-15, V, np.nan)

def chsh_theory(g, T, nth, N_k, eta_bs=0.5):
    return 2*np.sqrt(2)*visibility_theory(g, T, nth, N_k, eta_bs)

# SIMULATION
# Keep your existing definitions ABOVE this section:
# a_mode, hv_pairs, stokes_ops, one_photon_arm_projector,
# postselect_one_each_arm, polarization_moments, analyzer_vector,
# single_from_moments, coincidence_from_moments, visibility_sim,
# chsh_max_sim, simulate_case

# COMMON PHYSICAL PARAMETER GRID
g_pol = 0.04
eta_bs = 0.5
beta = 0.0
N_k = 0
theta_sim = np.linspace(0, 180, 19)

cases = [
    (0, 1.00, 0.00),
    (0, 0.85, 0.02)
]

Nk_scan = [0]
nth_scan = [0.00, 0.02, 0.04, 0.10]
T_scan = np.linspace(0.0, 1.0, 15)

# Helpers
def physical_key(Nk, T, nth):
    return int(Nk), round(float(T), 12), round(float(nth), 12)

def observable_key(*args):
    out = []
    for x in args:
        if isinstance(x, (float, np.floating)):
            out.append(round(float(x), 12))
        else:
            out.append(x)
    return tuple(out)

# Run ALL Fig. 2 simulation points for one Fock dimension
def run_polarization_for_dimension(d, n_jobs=1):
    parameter_keys = set()
    for Nk, T, nth in cases:
        parameter_keys.add(physical_key(Nk, T, nth))
    for Nk in Nk_scan:
        for nth in nth_scan:
            for T in T_scan:
                parameter_keys.add(physical_key(Nk, T, nth))
    parameter_keys = sorted(parameter_keys)
    print(f"d = {d}: {len(parameter_keys)} unique physical simulations")

    if n_jobs == 1:
        result_list = [simulate_case(d, Nk, g_pol, T, nth, eta_bs) for Nk, T, nth in parameter_keys]
    else:
        result_list = Parallel(n_jobs=n_jobs, backend="loky")(
            delayed(simulate_case)(d, Nk, g_pol, T, nth, eta_bs)
            for Nk, T, nth in parameter_keys
        )

    simulation_map = {key: result for key, result in zip(parameter_keys, result_list)}

    points = {"singles": {}, "coincidence": {}, "visibility": {}, "chsh": {}}

    # 1. Singles
    for Nk, T, nth in cases:
        r = simulation_map[physical_key(Nk, T, nth)]
        for alpha in theta_sim:
            sim = single_from_moments(r["M_raw"], alpha, arm="idler", outcome="+")
            th = singles_theory(g_pol, T, nth, Nk, eta_bs)["idler"]
            key = observable_key(Nk, T, nth, alpha, "idler")
            points["singles"][key] = {
                "sim": float(np.real(sim)),
                "th": float(np.real(th)),
                "Nk": Nk,
                "T": T,
                "nth": nth,
                "alpha": alpha
            }

    # 2. Coincidences
    for Nk, T, nth in cases:
        r = simulation_map[physical_key(Nk, T, nth)]
        M_pair = r["M_pair"]
        p_pair = r["p_pair"]
        if M_pair is None:
            continue

        for alpha in theta_sim:
            Cth = coincidence_theory(alpha, beta, g_pol, T, nth, Nk, eta_bs)

            HH_sim = p_pair*coincidence_from_moments(M_pair, alpha, beta, "+", "+")
            key_HH = observable_key(Nk, T, nth, alpha, "HH")
            points["coincidence"][key_HH] = {
                "sim": float(np.real(HH_sim)),
                "th": float(np.real(Cth["HH"])),
                "Nk": Nk,
                "T": T,
                "nth": nth,
                "alpha": alpha,
                "channel": "HH"
            }

            HV_sim = p_pair*coincidence_from_moments(M_pair, alpha, beta, "+", "-")
            key_HV = observable_key(Nk, T, nth, alpha, "HV")
            points["coincidence"][key_HV] = {
                "sim": float(np.real(HV_sim)),
                "th": float(np.real(Cth["HV"])),
                "Nk": Nk,
                "T": T,
                "nth": nth,
                "alpha": alpha,
                "channel": "HV"
            }

    # 3. Visibility
    for Nk in Nk_scan:
        for nth in nth_scan:
            for T in T_scan:
                r = simulation_map[physical_key(Nk, T, nth)]
                Vsim = r["V"]
                Vth = visibility_theory(g_pol, T, nth, Nk, eta_bs)
                key = observable_key(Nk, T, nth)
                points["visibility"][key] = {
                    "sim": float(Vsim) if np.isfinite(Vsim) else np.nan,
                    "th": float(Vth) if np.isfinite(Vth) else np.nan,
                    "Nk": Nk,
                    "T": T,
                    "nth": nth
                }

    # 4. Conditional CHSH
    for Nk in Nk_scan:
        for nth in nth_scan:
            for T in T_scan:
                r = simulation_map[physical_key(Nk, T, nth)]
                Ssim = r["S"]
                Sth = chsh_theory(g_pol, T, nth, Nk, eta_bs)
                key = observable_key(Nk, T, nth)
                points["chsh"][key] = {
                    "sim": float(Ssim) if np.isfinite(Ssim) else np.nan,
                    "th": float(Sth) if np.isfinite(Sth) else np.nan,
                    "Nk": Nk,
                    "T": T,
                    "nth": nth
                }

    return points

# Get finite matched arrays
def finite_observable_arrays(obs_points):
    sim = []
    th = []
    keys = []
    for key, v in obs_points.items():
        s = v["sim"]
        t = v["th"]
        if np.isfinite(s) and np.isfinite(t):
            sim.append(s)
            th.append(t)
            keys.append(key)
    return np.asarray(sim, dtype=float), np.asarray(th, dtype=float), keys

# Observable scale
def observable_scale(obs_points):
    _, th, _ = finite_observable_arrays(obs_points)
    if len(th) == 0:
        return np.nan
    scale = np.max(np.abs(th))
    if scale < 1e-15:
        return 1.0
    return scale

# Build convergence table
def make_polarization_convergence_table(results_by_d, Nk=0, g=g_pol, nth_max=max(nth_scan)):
    d_sorted = sorted(results_by_d)
    observables = ["singles", "coincidence", "visibility", "chsh"]
    reference_d = d_sorted[0]
    scales = {obs: observable_scale(results_by_d[reference_d][obs]) for obs in observables}
    rows = []

    for j, d in enumerate(d_sorted):
        current = results_by_d[d]
        row = {"d": d}

        for obs in observables:
            sim, th, keys = finite_observable_arrays(current[obs])
            err = np.abs(sim-th)
            scale = scales[obs]

            row[f"max_error_{obs}"] = np.max(err) if len(err) else np.nan
            row[f"RMSE_{obs}"] = np.sqrt(np.mean(err**2)) if len(err) else np.nan
            row[f"norm_max_error_{obs}"] = np.max(err)/scale if len(err) else np.nan
            row[f"norm_RMSE_{obs}"] = np.sqrt(np.mean(err**2))/scale if len(err) else np.nan

            if len(err):
                idx = np.argmax(err)
                row[f"worst_key_{obs}"] = keys[idx]
            else:
                row[f"worst_key_{obs}"] = None

        if j == 0:
            for obs in observables:
                row[f"max_step_{obs}"] = np.nan
                row[f"norm_max_step_{obs}"] = np.nan
        else:
            d_prev = d_sorted[j-1]
            previous = results_by_d[d_prev]

            for obs in observables:
                common = sorted(set(current[obs].keys()) & set(previous[obs].keys()))
                differences = []

                for key in common:
                    x = current[obs][key]["sim"]
                    y = previous[obs][key]["sim"]
                    if np.isfinite(x) and np.isfinite(y):
                        differences.append(abs(x-y))

                differences = np.asarray(differences, dtype=float)
                scale = scales[obs]
                row[f"max_step_{obs}"] = np.max(differences) if len(differences) else np.nan
                row[f"norm_max_step_{obs}"] = np.max(differences)/scale if len(differences) else np.nan

        # A-priori Fock-tail diagnostics
        M = 1 + 2*Nk
        eps_pair = np.tanh(g)**(2*d)
        eps_spdc_total = 1-(1-eps_pair)**M
        N_post = 4 + 8*Nk
        qth = nth_max/(1+nth_max)
        eps_env_total = 1-(1-qth**d)**N_post

        row["eps_SPDC"] = eps_spdc_total
        row["eps_env_postNPBS"] = eps_env_total
        rows.append(row)

    return pd.DataFrame(rows)

# Compact convergence figure
def plot_polarization_convergence(df):
    d = df["d"].values
    fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.5))

    observable_info = [
        ("singles", "Singles", "o"),
        ("coincidence", "Coincidences", "s"),
        ("visibility", r"$V_{\rm pol}$", "^"),
        ("chsh", r"$S_{\rm CHSH}$", "D")
    ]

    # (a) Reduced-theory -- simulation discrepancy
    for obs, label, marker in observable_info:
        y = df[f"norm_max_error_{obs}"].values
        axes[0].plot(d, np.maximum(y, 1e-16), marker=marker, linewidth=1.4, markersize=5.5, label=label)

    axes[0].set_yscale("log")
    axes[0].set_xlabel(r"Local Fock dimension $d$", fontsize=12)
    axes[0].set_ylabel("Normalized $\\max|\\mathcal{O}^{(d)}-\\mathcal{O}_{\\rm th}|$", fontsize=12)
    axes[0].set_title("Theory-simulation discrepancy", fontsize=12)
    axes[0].set_xticks(d)
    axes[0].xaxis.set_minor_locator(AutoMinorLocator(4))

    # (b) Successive-cutoff numerical convergence
    valid = np.isfinite(df["norm_max_step_singles"].values)
    d_valid = d[valid]

    for obs, label, marker in observable_info:
        y = df.loc[valid, f"norm_max_step_{obs}"].values
        axes[1].plot(d_valid, np.maximum(y, 1e-16), marker=marker, linewidth=1.4, markersize=5.5, label=label)

    axes[1].set_yscale("log")
    axes[1].set_xlabel("Successive local Fock cutoff", fontsize=12)
    axes[1].set_ylabel("Normalized $\\max|\\mathcal{O}^{(d)}-\\mathcal{O}^{(d_{\\rm prev})}|$", fontsize=12)
    axes[1].set_title("Successive-cutoff change", fontsize=12)
    axes[1].set_xticks(d_valid)

    transition_labels = []
    for upper_d in d_valid:
        idx = np.where(d == upper_d)[0][0]
        lower_d = d[idx-1]
        transition_labels.append(fr"${lower_d}" r"\!\to\!" fr"{upper_d}$")

    axes[1].set_xticklabels(transition_labels)
    axes[1].xaxis.set_minor_locator(AutoMinorLocator(4))

    # Identical y-axis limits
    all_values = []
    for obs, _, _ in observable_info:
        x = df[f"norm_max_error_{obs}"].values
        x = x[np.isfinite(x) & (x > 0)]
        all_values.extend(x.tolist())
        x = df[f"norm_max_step_{obs}"].values
        x = x[np.isfinite(x) & (x > 0)]
        all_values.extend(x.tolist())

    if len(all_values):
        ymin = 10**(np.floor(np.log10(np.min(all_values)))-0.25)
        ymax = 10**(np.ceil(np.log10(np.max(all_values))))
        for ax in axes:
            ax.set_ylim(ymin, ymax)

    # Common formatting
    for ax, panel in zip(axes, ["(a)", "(b)"]):
        ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2, 10)*0.1, numticks=100))
        ax.yaxis.set_minor_formatter(NullFormatter())
        ax.text(-0.15, 1.06, panel, transform=ax.transAxes, fontsize=14, fontweight="bold", va="bottom", clip_on=False)
        ax.tick_params(which="major", direction="in", length=6, top=True, right=True, labelsize=10)
        ax.tick_params(which="minor", direction="in", length=3, top=True, right=True)
        ax.grid(True, which="major", linestyle="--", linewidth=0.6, alpha=0.40)
        ax.grid(True, which="minor", linestyle="--", linewidth=0.35, alpha=0.18)
        ax.legend(frameon=False, fontsize=9)

    plt.tight_layout()
    plt.savefig("images/polarization_fock_convergence.png", dpi=400, bbox_inches="tight")
    plt.show()

# Compact LaTeX table
def print_polarization_latex_table(df):
    compact = pd.DataFrame({
        "d": df["d"],
        "Delta_single": df["norm_max_error_singles"],
        "Delta_coin": df["norm_max_error_coincidence"],
        "Delta_V": df["norm_max_error_visibility"],
        "Delta_S": df["norm_max_error_chsh"],
        "delta_single": df["norm_max_step_singles"],
        "delta_coin": df["norm_max_step_coincidence"],
        "delta_V": df["norm_max_step_visibility"],
        "delta_S": df["norm_max_step_chsh"],
        "eps_SPDC": df["eps_SPDC"],
        "eps_env": df["eps_env_postNPBS"]
    })
    print(compact.to_latex(index=False, float_format=lambda x: f"{x:.3e}"))

# Optional: print worst theory-disagreement points
def print_worst_points(df):
    observables = ["singles", "coincidence", "visibility", "chsh"]
    for _, row in df.iterrows():
        print(f"\nd = {int(row['d'])}")
        for obs in observables:
            print(f"  {obs:12s}: {row[f'worst_key_{obs}']}")

# RUN d = 2,...,6
d_values = [2, 3, 4, 5, 6]
measurement_results_by_d = {}

for d in d_values:
    print("\n" + "="*60)
    print(f"Running polarization convergence: d = {d}")
    print("="*60)
    measurement_results_by_d[d] = run_polarization_for_dimension(d=d, n_jobs=1)

# CONVERGENCE TABLE
df_pol_conv = make_polarization_convergence_table(
    measurement_results_by_d,
    Nk=N_k,
    g=g_pol,
    nth_max=max(nth_scan)
)

# Print full table
print("\nFULL POLARIZATION CONVERGENCE TABLE\n")
print(df_pol_conv.to_string(index=False))

# Save complete CSV
df_pol_conv.to_csv("images/polarization_fock_convergence_table.csv", index=False)

# Print compact LaTeX table
print("\nCOMPACT LATEX TABLE\n")
print_polarization_latex_table(df_pol_conv)

# Print worst physical parameter points
print("\nWORST THEORY-DISAGREEMENT POINTS\n")
print_worst_points(df_pol_conv)

# Plot
plot_polarization_convergence(df_pol_conv)

## 19. Interpretation and reproducibility checklist

Before interpreting a theory–simulation residual, verify numerical convergence with respect to $d$.

- The **state-level Gaussian theory** is exact under the stated factorized-source and mode-separable thermal-loss assumptions.
- The **multimode state-level extension** is multiplicative over $M=1+2N_k$ independent pairs.
- The **polarization-count theory** is a reduced weak-gain model. A residual can remain after the finite-Fock simulation has converged.
- The full finite-Fock simulation retains higher photon-number sectors up to the selected cutoff.
- The physical probability of generating at least two pairs from one TMSV source is

$$
P_{n\ge2}=\tanh^4 g,
$$

which is distinct from numerical Fock-space truncation.
- Conditional CHSH quantities are evaluated after one-photon-per-arm postselection and are not, by themselves, loophole-free Bell-test statistics.

### Selected references

Key sources used in the methodology include:

- C. K. Law, I. A. Walmsley, and J. H. Eberly, *Phys. Rev. Lett.* **84**, 5304 (2000).
- C. Weedbrook *et al.*, *Rev. Mod. Phys.* **84**, 621 (2012).
- G. Adesso, S. Ragy, and A. R. Lee, *Open Syst. Inf. Dyn.* **21**, 1440001 (2014).
- L. Banchi, S. L. Braunstein, and S. Pirandola, *Phys. Rev. Lett.* **115**, 260501 (2015).
- R. Horodecki, P. Horodecki, and M. Horodecki, *Phys. Lett. A* **200**, 340 (1995).
- M. P. Woods, M. Cramer, and M. B. Plenio, *Phys. Rev. Lett.* **115**, 130401 (2015).
- G. Vallone, G. Cariolaro, and G. Pierobon, *Phys. Rev. A* **99**, 023817 (2019).